# 模块概述

WtBtPorter 是 WonderTrader 回测模块的 C 接口导出模块，用于外部语言（如 Python、C#、Java 等）与 WonderTrader C++ 回测引擎进行交互。该模块通过 C 接口和回调函数机制，实现了跨语言调用和事件驱动的回测编程模型。

主要包括：
- C 接口导出层：提供统一的 C 风格函数接口，供外部语言调用
- 回测运行器：管理整个回测系统的运行时环境
- 扩展模拟器：适配器类，将回测事件转发给外部语言回调
- 数据回放支持：通过历史数据回放器驱动回测执行
- 数据加载支持：支持外部数据源加载历史数据
- 回调函数管理：统一管理各种策略模拟器的事件回调

1. **C接口导出层**（WtBtPorter.h/cpp）：
   - 提供 C 风格导出函数，供外部语言调用
   - 包括策略回调注册、回测引擎初始化配置、策略模拟器创建、交易操作、数据查询等接口
   - 使用单例模式管理 WtBtRunner 对象
   - 通过上下文句柄（CtxHandler）管理策略模拟器实例，避免直接暴露 C++ 对象指针

2. **回测运行器层**（WtBtRunner）：
   - WtBtRunner：回测运行器核心，管理整个回测系统的生命周期
   - 实现 IBtDataLoader 接口，支持外部数据加载器
   - 管理策略模拟器（CTA、HFT、SEL模拟器）的创建和映射
   - 管理回调函数的注册和事件转发
   - 通过 HisDataReplayer 回放历史数据，驱动策略执行
   - 支持同步和异步两种回测模式

3. **扩展模拟器层**（ExpCtaMocker + ExpHftMocker + ExpSelMocker）：
   - ExpCtaMocker：CTA策略扩展模拟器，继承自CtaMocker，转发CTA策略回测事件
   - ExpHftMocker：HFT策略扩展模拟器，继承自HftMocker，转发HFT策略回测事件
   - ExpSelMocker：SEL策略扩展模拟器，继承自SelMocker，转发SEL策略回测事件
   - 作为适配器，将回测过程中的各种事件（初始化、交易日事件、Tick更新、K线闭合、策略计算等）转发给外部语言

4. **定义层**（PorterDefs.h）：
   - 定义回调函数类型、事件常量、上下文句柄类型等基础定义
   - 提供统一的类型定义，供C接口和外部语言使用
   - 与 WtPorter 模块共享相同的定义文件

5. **核心回测引擎层**（依赖WtBtCore模块）：
   - HisDataReplayer：历史数据回放器，负责回放历史数据并驱动策略执行
   - CtaMocker：CTA策略模拟器基类，提供CTA策略的回测环境
   - HftMocker：HFT策略模拟器基类，提供HFT策略的回测环境
   - SelMocker：SEL策略模拟器基类，提供SEL策略的回测环境
   - EventNotifier：事件通知器，支持事件通知功能

# 层次关系图
```mermaid
graph TB
    %% 样式定义
    classDef cInterfaceClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef runnerClass fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef mockerClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef replayerClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;
    classDef defClass fill:#fffde7,stroke:#f57f17,stroke-width:2px,color:#000;
    classDef baseClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;
    classDef interfaceClass fill:#e0f2f1,stroke:#004d40,stroke-width:2px,color:#000;

    %% C接口导出层
    subgraph CInterfaceLayer["C接口导出层 - 外部语言绑定"]
        direction TB
        WtBtPorter["WtBtPorter.h/cpp<br/>回测C接口导出<br/>• 策略回调注册<br/>• 回测引擎初始化配置<br/>• 策略模拟器创建<br/>• 交易操作接口<br/>• 数据查询接口<br/>• 数据推送接口"]:::cInterfaceClass
    end

    %% 回测运行器层
    subgraph RunnerLayer["回测运行器层 - 核心调度"]
        direction TB
        WtBtRunner["WtBtRunner<br/>回测运行器<br/>• 实现IBtDataLoader<br/>• 策略模拟器管理<br/>• 回调函数管理<br/>• 数据加载管理<br/>• 历史数据回放控制<br/>• 事件转发"]:::runnerClass
    end

    %% 扩展模拟器层
    subgraph MockerLayer["扩展模拟器层 - 事件适配"]
        direction TB
        ExpCtaMocker["ExpCtaMocker<br/>CTA策略扩展模拟器<br/>• 继承CtaMocker<br/>• 事件转发适配<br/>• 初始化/Tick/Bar/Calc事件"]:::mockerClass
        ExpHftMocker["ExpHftMocker<br/>HFT策略扩展模拟器<br/>• 继承HftMocker<br/>• 事件转发适配<br/>• 订单/成交/Level2事件"]:::mockerClass
        ExpSelMocker["ExpSelMocker<br/>SEL策略扩展模拟器<br/>• 继承SelMocker<br/>• 事件转发适配<br/>• 调度/Tick/Bar事件"]:::mockerClass
    end

    %% 定义层
    subgraph DefLayer["定义层 - 基础类型"]
        direction TB
        PorterDefs["PorterDefs.h<br/>Porter模块定义<br/>• 回调函数类型<br/>• 事件常量<br/>• 上下文句柄类型<br/>• 数据类型定义"]:::defClass
    end

    %% 核心回测引擎层（依赖）
    subgraph BtCoreLayer["核心回测引擎层 - 依赖WtBtCore模块"]
        direction TB
        HisDataReplayer["HisDataReplayer<br/>历史数据回放器<br/>• 历史数据回放<br/>• 时间推进<br/>• 事件触发<br/>• 数据分发"]:::replayerClass
        EventNotifier["EventNotifier<br/>事件通知器<br/>• 事件通知<br/>• 消息队列集成"]:::replayerClass
    end

    %% 模拟器基类（引用）
    subgraph BaseMockerLayer["模拟器基类 - 来自WtBtCore模块"]
        direction TB
        CtaMocker["CtaMocker<br/>CTA策略模拟器基类<br/>• 回测环境<br/>• 仓位管理<br/>• 成交撮合"]:::baseClass
        HftMocker["HftMocker<br/>HFT策略模拟器基类<br/>• 回测环境<br/>• 订单管理<br/>• Level2支持"]:::baseClass
        SelMocker["SelMocker<br/>SEL策略模拟器基类<br/>• 回测环境<br/>• 选股逻辑<br/>• 调度机制"]:::baseClass
    end

    %% 接口类（引用）
    subgraph InterfaceLayer["接口类 - 来自Includes模块"]
        direction TB
        IBtDataLoader["IBtDataLoader<br/>回测数据加载器接口<br/>• 加载K线数据<br/>• 加载Tick数据<br/>• 加载复权因子"]:::interfaceClass
        IDataSink["IDataSink<br/>数据接收接口<br/>• 接收Tick数据<br/>• 接收K线数据"]:::baseClass
    end

    %% C接口到运行器的关系
    WtBtPorter -->|"调用"| WtBtRunner
    
    %% 运行器到回放器的关系
    WtBtRunner -->|"使用"| HisDataReplayer
    WtBtRunner -->|"使用"| EventNotifier
    
    %% 运行器到扩展模拟器的关系
    WtBtRunner -->|"创建"| ExpCtaMocker
    WtBtRunner -->|"创建"| ExpHftMocker
    WtBtRunner -->|"创建"| ExpSelMocker
    
    %% 扩展模拟器继承关系
    ExpCtaMocker -.->|"继承"| CtaMocker
    ExpHftMocker -.->|"继承"| HftMocker
    ExpSelMocker -.->|"继承"| SelMocker
    
    %% 运行器实现接口关系
    WtBtRunner -.->|"实现"| IBtDataLoader
    
    %% 模拟器实现接口关系
    CtaMocker -.->|"实现"| IDataSink
    HftMocker -.->|"实现"| IDataSink
    SelMocker -.->|"实现"| IDataSink
    
    %% 回放器到模拟器的关系
    HisDataReplayer -->|"推送数据"| CtaMocker
    HisDataReplayer -->|"推送数据"| HftMocker
    HisDataReplayer -->|"推送数据"| SelMocker
    
    %% 扩展模拟器到回放器的关系
    ExpCtaMocker -->|"使用"| HisDataReplayer
    ExpHftMocker -->|"使用"| HisDataReplayer
    ExpSelMocker -->|"使用"| HisDataReplayer
    
    %% 回调函数流
    WtBtRunner -.->|"注册回调"| PorterDefs
    ExpCtaMocker -.->|"事件回调"| WtBtRunner
    ExpHftMocker -.->|"事件回调"| WtBtRunner
    ExpSelMocker -.->|"事件回调"| WtBtRunner
    
    %% 外部语言交互
    ExternalLang["外部语言<br/>(Python/C#等)"] -.->|"调用C接口"| WtBtPorter
    ExternalLang -.->|"注册回调"| WtBtPorter
    ExternalLang -.->|"推送数据"| WtBtPorter
    WtBtPorter -.->|"事件通知"| ExternalLang
    
    %% 数据流
    ExternalLang -.->|"数据加载"| WtBtRunner
    WtBtRunner -.->|"数据加载"| IBtDataLoader
    HisDataReplayer -->|"数据回放"| ExpCtaMocker
    HisDataReplayer -->|"数据回放"| ExpHftMocker
    HisDataReplayer -->|"数据回放"| ExpSelMocker
    
    %% 应用样式
    class WtBtPorter cInterfaceClass
    class WtBtRunner runnerClass
    class ExpCtaMocker,ExpHftMocker,ExpSelMocker mockerClass
    class HisDataReplayer,EventNotifier replayerClass
    class PorterDefs defClass
    class CtaMocker,HftMocker,SelMocker,IDataSink baseClass
    class IBtDataLoader interfaceClass
```

# 基础类型定义 PorterDefs.h

# 回测C接口导出 WtBtPorter.h/cpp

## 回调函数注册接口 

### 注册引擎事件回调函数 register_evt_callback
```cpp
/**
 * @brief 注册引擎事件回调函数
 * 
 * 将外部语言注册的事件回调函数传递给WtBtRunner
 * 
 * @param cbEvt 事件回调函数指针
 */
void register_evt_callback(FuncEventCallback cbEvt)
{
	getRunner().registerEvtCallback(cbEvt);  // 调用WtBtRunner的注册事件回调方法
}
```

### 注册CTA策略回调函数 register_cta_callbacks
```cpp
/**
 * @brief 注册CTA策略回调函数
 * 
 * 将外部语言注册的CTA策略回调函数传递给WtBtRunner
 * 
 * @param cbInit 策略初始化回调函数
 * @param cbTick Tick更新回调函数
 * @param cbCalc 策略计算回调函数
 * @param cbBar K线闭合回调函数
 * @param cbSessEvt 交易日事件回调函数
 * @param cbCalcDone 策略计算完成回调函数（可选）
 * @param cbCondTrigger 条件单触发回调函数（可选）
 */
void register_cta_callbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraCalcCallback cbCalc, 
	FuncStraBarCallback cbBar, FuncSessionEvtCallback cbSessEvt, FuncStraCalcCallback cbCalcDone/* = NULL*/, FuncStraCondTriggerCallback cbCondTrigger)
{
	getRunner().registerCtaCallbacks(cbInit, cbTick, cbCalc, cbBar, cbSessEvt, cbCalcDone, cbCondTrigger);  // 调用WtBtRunner的注册CTA回调方法
}
```

### 注册SEL策略回调函数 register_sel_callbacks
```cpp
/**
 * @brief 注册SEL策略回调函数
 * 
 * 将外部语言注册的SEL策略回调函数传递给WtBtRunner
 * 
 * @param cbInit 策略初始化回调函数
 * @param cbTick Tick更新回调函数
 * @param cbCalc 策略计算回调函数
 * @param cbBar K线闭合回调函数
 * @param cbSessEvt 交易日事件回调函数
 * @param cbCalcDone 策略计算完成回调函数（可选）
 */
void register_sel_callbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraCalcCallback cbCalc, 
	FuncStraBarCallback cbBar, FuncSessionEvtCallback cbSessEvt, FuncStraCalcCallback cbCalcDone/* = NULL*/)
{
	getRunner().registerSelCallbacks(cbInit, cbTick, cbCalc, cbBar, cbSessEvt, cbCalcDone);  // 调用WtBtRunner的注册SEL回调方法
}
```

### 注册HFT策略回调函数 register_hft_callbacks
```cpp
/**
 * @brief 注册HFT策略回调函数
 * 
 * 将外部语言注册的HFT策略回调函数传递给WtBtRunner
 * 
 * @param cbInit 策略初始化回调函数
 * @param cbTick Tick更新回调函数
 * @param cbBar K线闭合回调函数
 * @param cbChnl 交易通道事件回调函数
 * @param cbOrd 订单状态变化回调函数
 * @param cbTrd 成交回报回调函数
 * @param cbEntrust 委托回报回调函数
 * @param cbOrdDtl 订单明细更新回调函数
 * @param cbOrdQue 订单队列更新回调函数
 * @param cbTrans 逐笔成交更新回调函数
 * @param cbSessEvt 交易日事件回调函数
 */
void register_hft_callbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraBarCallback cbBar,
	FuncHftChannelCallback cbChnl, FuncHftOrdCallback cbOrd, FuncHftTrdCallback cbTrd, FuncHftEntrustCallback cbEntrust,
	FuncStraOrdDtlCallback cbOrdDtl, FuncStraOrdQueCallback cbOrdQue, FuncStraTransCallback cbTrans, FuncSessionEvtCallback cbSessEvt)
{
	getRunner().registerHftCallbacks(cbInit, cbTick, cbBar, cbChnl, cbOrd, cbTrd, cbEntrust, cbOrdDtl, cbOrdQue, cbTrans, cbSessEvt);  // 调用WtBtRunner的注册HFT回调方法
}
```

### 注册外部数据加载器 register_ext_data_loader
```cpp
/**
 * @brief 注册外部数据加载器
 * 
 * 将外部语言注册的数据加载器回调函数传递给WtBtRunner
 * 
 * @param fnlBarLoader 加载最终K线数据的回调函数（已复权）
 * @param rawBarLoader 加载原始K线数据的回调函数（未复权）
 * @param fctLoader 加载复权因子的回调函数
 * @param tickLoader 加载原始Tick数据的回调函数
 * @param bAutoTrans 是否自动转储数据到本地缓存
 */
void register_ext_data_loader(FuncLoadFnlBars fnlBarLoader, FuncLoadRawBars rawBarLoader, FuncLoadAdjFactors fctLoader, FuncLoadRawTicks tickLoader, bool bAutoTrans)
{
	getRunner().registerExtDataLoader(fnlBarLoader, rawBarLoader, fctLoader, tickLoader, bAutoTrans);  // 调用WtBtRunner的注册外部数据加载器方法
}
```

## 数据推送接口

### 推送原始K线数据 feed_raw_bars
```cpp
/**
 * @brief 推送原始K线数据
 * 
 * 将外部数据源的原始K线数据推送到WtBtRunner中
 * 
 * @param bars K线数据数组指针
 * @param count K线数据条数
 */
void feed_raw_bars(WTSBarStruct* bars, WtUInt32 count)
{
	getRunner().feedRawBars(bars, count);  // 调用WtBtRunner的推送原始K线数据方法
}
```

### 推送原始Tick数据 feed_raw_ticks
```cpp
/**
 * @brief 推送原始Tick数据
 * 
 * 将外部数据源的原始Tick数据推送到WtBtRunner中
 * 
 * @param ticks Tick数据数组指针
 * @param count Tick数据条数
 */
void feed_raw_ticks(WTSTickStruct* ticks, WtUInt32 count)
{
	getRunner().feedRawTicks(ticks, count);  // 调用WtBtRunner的推送原始Tick数据方法
}
```

### 推送复权因子数据 feed_adj_factors
```cpp
/**
 * @brief 推送复权因子数据
 * 
 * 将外部数据源的复权因子数据推送到WtRtRunner中
 * 
 * @param stdCode 标准合约代码
 * @param dates 日期数组指针（格式：YYYYMMDD）
 * @param factors 复权因子数组指针
 * @param count 数据条数
 */
void feed_adj_factors(WtString stdCode, WtUInt32* dates, double* factors, WtUInt32 count)
{
	getRunner().feedAdjFactors(stdCode, (uint32_t*)dates, factors, count);
}
```

## 回测引擎管理接口

### 初始化回测引擎 init_backtest
```cpp
/**
 * @brief 初始化回测引擎
 * 
 * 初始化回测引擎，设置日志配置和输出目录
 * 使用静态变量确保只初始化一次
 * 
 * @param logProfile 日志配置文件路径或配置内容
 * @param isFile true表示logProfile是文件路径，false表示logProfile是配置内容
 * @param outDir 回测结果输出目录
 */
void init_backtest(const char* logProfile, bool isFile, const char* outDir)
{
	static bool inited = false;  // 静态变量，确保只初始化一次

	if (inited)  // 如果已经初始化过，直接返回
		return;

	getRunner().init(logProfile, isFile, outDir);  // 调用WtBtRunner的初始化方法

	inited = true;  // 标记为已初始化
}
```

### 配置回测引擎 config_backtest
```cpp
/**
 * @brief 配置回测引擎
 * 
 * 加载并应用配置文件，配置回测引擎的各种参数
 * 使用静态变量确保只配置一次
 * 
 * @param cfgfile 配置文件路径或配置内容（JSON格式）
 * @param isFile true表示cfgfile是文件路径，false表示cfgfile是配置内容
 */
void config_backtest(const char* cfgfile, bool isFile)
{
	static bool inited = false;  // 静态变量，确保只配置一次

	if (inited)  // 如果已经配置过，直接返回
		return;

	if (strlen(cfgfile) == 0)  // 如果配置文件路径为空，使用默认配置文件
		getRunner().config("configbt.json", true);  // 使用默认配置文件"configbt.json"
	else
		getRunner().config(cfgfile, isFile);  // 调用WtBtRunner的配置方法
}
```

### 设置回测时间范围 set_time_range
```cpp
/**
 * @brief 设置回测时间范围
 * 
 * 设置回测的开始时间和结束时间
 * 
 * @param stime 开始时间戳（格式：YYYYMMDDHHMMSS）
 * @param etime 结束时间戳（格式：YYYYMMDDHHMMSS）
 */
void set_time_range(WtUInt64 stime, WtUInt64 etime)
{
	getRunner().set_time_range(stime, etime);  // 调用WtBtRunner的设置时间范围方法
}
```

### 启用/禁用Tick回放 enable_tick
```cpp
/**
 * @brief 启用/禁用Tick回放
 * 
 * 控制回测引擎是否回放Tick数据（如果禁用，则只回放K线数据）
 * 
 * @param bEnabled true表示启用Tick回放，false表示禁用Tick回放（默认启用）
 */
void enable_tick(bool bEnabled /* = true */)
{
	getRunner().enable_tick(bEnabled);  // 调用WtBtRunner的启用/禁用Tick回放方法
}
```

### 运行回测 run_backtest
```cpp
/**
 * @brief 运行回测
 * 
 * 启动回测引擎，开始执行回测
 * 
 * @param bNeedDump true表示需要输出回测结果到文件，false表示不输出
 * @param bAsync true表示异步运行（函数立即返回），false表示同步运行（函数阻塞直到回测完成）
 */
void run_backtest(bool bNeedDump, bool bAsync)
{
	getRunner().run(bNeedDump, bAsync);  // 调用WtBtRunner的运行方法
}
```

### 停止回测 stop_backtest
```cpp
/**
 * @brief 停止回测
 * 
 * 停止正在运行的回测（异步回测模式下使用）
 */
void stop_backtest()
{
	getRunner().stop();  // 调用WtBtRunner的停止方法
}
```

### 释放回测引擎 release_backtest
```cpp
/**
 * @brief 释放回测引擎
 * 
 * 清理资源，释放回测引擎占用的资源
 */
void release_backtest()
{
	getRunner().release();  // 调用WtBtRunner的释放方法
}
```

### 清空缓存 clear_cache
```cpp
/**
 * @brief 清空缓存
 * 
 * 清空回测引擎的数据缓存
 */
void clear_cache()
{
	getRunner().clear_cache();  // 调用WtBtRunner的清空缓存方法
}
```

## 策略模拟器初始化接口

### 初始化CTA策略模拟器 init_cta_mocker
```cpp
/**
 * @brief 初始化CTA策略模拟器
 * 
 * 创建一个新的CTA策略模拟器实例
 * 
 * @param name 策略名称
 * @param slippage 滑点设置（单位：最小变动价位，0表示不设置滑点）
 * @param hook true表示启用钩子模式（用于调试），false表示正常模式（默认false）
 * @param persistData true表示持久化策略数据（程序重启后可恢复），false表示不持久化（默认true）
 * @param bIncremental true表示增量回测模式，false表示全量回测模式（默认false）
 * @param bRatioSlp true表示使用比例滑点（滑点=价格*比例），false表示使用固定滑点（默认false）
 * @return 策略上下文句柄
 */
CtxHandler init_cta_mocker(const char* name, int slippage/* = 0*/, bool hook/* = false*/, bool persistData/* = true*/, bool bIncremental/* = false*/, bool bRatioSlp/* = false*/)
{
	return getRunner().initCtaMocker(name, slippage, hook, persistData, bIncremental, bRatioSlp);  // 调用WtBtRunner的初始化CTA模拟器方法
}
```

### 初始化HFT策略模拟器 init_hft_mocker
```cpp
/**
 * @brief 初始化HFT策略模拟器
 * 
 * 创建一个新的HFT策略模拟器实例
 * 
 * @param name 策略名称
 * @param hook true表示启用钩子模式（用于调试），false表示正常模式（默认false）
 * @return 策略上下文句柄
 */
CtxHandler init_hft_mocker(const char* name, bool hook/* = false*/)
{
	return getRunner().initHftMocker(name, hook);  // 调用WtBtRunner的初始化HFT模拟器方法
}
```

### 初始化SEL策略模拟器 init_sel_mocker
```cpp
/**
 * @brief 初始化SEL策略模拟器
 * 
 * 创建一个新的选股策略模拟器实例
 * 
 * @param name 策略名称
 * @param date 策略开始日期（格式：YYYYMMDD）
 * @param time 策略开始时间（格式：HHMMSS）
 * @param period 策略执行周期（"d"-日线，"w"-周线，"m"-月线，"y"-年线）
 * @param trdtpl 交易模板名称（默认为"CHINA"，表示中国A股市场）
 * @param session 交易时段名称（默认为"TRADING"，表示交易时段）
 * @param slippage 滑点设置（单位：最小变动价位，0表示不设置滑点）
 * @param bRatioSlp true表示使用比例滑点，false表示使用固定滑点（默认false）
 * @return 策略上下文句柄
 */
CtxHandler init_sel_mocker(const char* name, WtUInt32 date, WtUInt32 time, const char* period, const char* trdtpl/* = "CHINA"*/, const char* session/* = "TRADING"*/, int slippage/* = 0*/, bool bRatioSlp/* = false*/)
{
	return getRunner().initSelMocker(name, date, time, period, trdtpl, session, slippage, bRatioSlp);  // 调用WtBtRunner的初始化SEL模拟器方法
}
```

## 工具接口

### 写入日志 write_log
```cpp
/**
 * @brief 写入日志
 * 
 * 向日志系统写入一条日志记录
 * 
 * @param level 日志级别（LOG_LEVEL_DEBUG、LOG_LEVEL_INFO、LOG_LEVEL_WARN、LOG_LEVEL_ERROR）
 * @param message 日志消息内容
 * @param catName 日志分类名称（可选，为空则使用默认分类）
 */
void write_log(WtUInt32 level, const char* message, const char* catName)
{
	if (strlen(catName) > 0)  // 如果指定了分类名称，使用分类日志
	{
		WTSLogger::log_raw_by_cat(catName, (WTSLogLevel)level, message);  // 按分类记录日志
	}
	else  // 否则使用默认日志
	{
		WTSLogger::log_raw((WTSLogLevel)level, message);  // 记录默认日志
	}
}
```

### 获取版本信息 get_version
```cpp
/**
 * @brief 获取版本信息
 * 
 * 获取WonderTrader框架的版本信息字符串，包含平台、版本号、编译日期和时间
 * 使用静态变量缓存版本信息，避免重复构建字符串
 * 
 * @return 版本信息字符串（格式：平台 版本号 Build@编译日期 编译时间）
 */
const char* get_version()
{
	static std::string _ver;  // 静态变量，缓存版本信息字符串
	if(_ver.empty())  // 如果版本信息尚未构建，则构建一次
	{
		_ver = PLATFORM_NAME;  // 平台名称（X64/X86/UNIX）
		_ver += " ";
		_ver += WT_VERSION;  // 版本号
		_ver += " Build@";
		_ver += __DATE__;  // 编译日期（宏定义）
		_ver += " ";
		_ver += __TIME__;  // 编译时间（宏定义）
	}
	return _ver.c_str();  // 返回C风格字符串
}
```

### 获取原始标准代码 get_raw_stdcode
```cpp
/**
 * @brief 获取原始标准代码
 * 
 * 将标准合约代码转换为原始合约代码（去除复权、主力等后缀）
 * 
 * @param stdCode 标准合约代码（如"SHFE.rb2305.HOT"）
 * @return 原始合约代码（如"SHFE.rb2305"）
 */
WtString get_raw_stdcode(const char* stdCode)
{
	return getRunner().get_raw_stdcode(stdCode);  // 调用WtBtRunner的获取原始标准代码方法
}
```

## CTA策略接口

### 交易操作接口

#### 开多仓 cta_enter_long
```cpp
/**
 * @brief 开多仓
 * 
 * 执行开多仓操作，买入指定数量的合约
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用，保留以保持接口一致性）
 * @param stdCode 标准合约代码
 * @param qty 开仓数量
 * @param userTag 用户标签（用于标识该笔交易）
 * @param limitprice 限价价格（0表示市价）
 * @param stopprice 止损价格（0表示不设置止损）
 */
void cta_enter_long(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag, double limitprice, double stopprice)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->stra_enter_long(stdCode, qty, userTag, limitprice, stopprice);  // 调用模拟器的开多仓方法
}
```

#### 平多仓 cta_exit_long
```cpp
/**
 * @brief 平多仓
 * 
 * 执行平多仓操作，卖出指定数量的多头持仓
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param qty 平仓数量
 * @param userTag 用户标签
 * @param limitprice 限价价格（0表示市价）
 * @param stopprice 止损价格（0表示不设置止损）
 */
void cta_exit_long(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag, double limitprice, double stopprice)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->stra_exit_long(stdCode, qty, userTag, limitprice, stopprice);  // 调用模拟器的平多仓方法
}
```

#### 开空仓 cta_enter_short
```cpp
/**
 * @brief 开空仓
 * 
 * 执行开空仓操作，卖出指定数量的合约
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param qty 开仓数量
 * @param userTag 用户标签
 * @param limitprice 限价价格（0表示市价）
 * @param stopprice 止损价格（0表示不设置止损）
 */
void cta_enter_short(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag, double limitprice, double stopprice)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->stra_enter_short(stdCode, qty, userTag, limitprice, stopprice);  // 调用模拟器的开空仓方法
}
```

#### 平空仓 cta_exit_short
```cpp
/**
 * @brief 平空仓
 * 
 * 执行平空仓操作，买入指定数量的空头持仓
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param qty 平仓数量
 * @param userTag 用户标签
 * @param limitprice 限价价格（0表示市价）
 * @param stopprice 止损价格（0表示不设置止损）
 */
void cta_exit_short(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag, double limitprice, double stopprice)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->stra_exit_short(stdCode, qty, userTag, limitprice, stopprice);  // 调用模拟器的平空仓方法
}
```

#### 设置目标持仓 cta_set_position
```cpp
/**
 * @brief 设置目标持仓
 * 
 * 设置指定合约的目标持仓数量，系统会自动计算需要开仓或平仓的数量并执行
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param qty 目标持仓数量（正数表示多头，负数表示空头，0表示平仓）
 * @param userTag 用户标签
 * @param limitprice 限价价格（0表示市价）
 * @param stopprice 止损价格（0表示不设置止损）
 */
void cta_set_position(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag, double limitprice, double stopprice)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->stra_set_position(stdCode, qty, userTag, limitprice, stopprice);  // 调用模拟器的设置目标持仓方法
}
```

### 持仓查询接口

#### 获取持仓数量 cta_get_position
```cpp
/**
 * @brief 获取持仓盈亏
 * 
 * 获取指定合约的持仓浮动盈亏
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 持仓盈亏金额（正数表示盈利，负数表示亏损，如果模拟器不存在则返回0）
 */
double cta_get_position_profit(CtxHandler cHandle, const char* stdCode)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_position_profit(stdCode);  // 调用模拟器的获取持仓盈亏方法
}
```

#### 获取持仓盈亏 cta_get_position_profit
```cpp
/**
 * @brief 获取持仓盈亏
 * 
 * 获取指定合约的持仓浮动盈亏
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 持仓盈亏金额（正数表示盈利，负数表示亏损，如果模拟器不存在则返回0）
 */
double cta_get_position_profit(CtxHandler cHandle, const char* stdCode)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_position_profit(stdCode);  // 调用模拟器的获取持仓盈亏方法
}
```

#### 获取持仓均价 cta_get_position_avgpx
```cpp
/**
 * @brief 获取持仓均价
 * 
 * 获取指定合约的持仓平均价格
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 持仓均价（如果模拟器不存在则返回0）
 */
double cta_get_position_avgpx(CtxHandler cHandle, const char* stdCode)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_position_avgpx(stdCode);  // 调用模拟器的获取持仓均价方法
}
```

#### 获取明细持仓的入场时间 cta_get_detail_entertime
```cpp
/**
 * @brief 获取明细持仓的入场时间
 * 
 * 获取指定合约和开仓标签对应的持仓明细的入场时间
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签（用于区分不同的开仓批次）
 * @return 入场时间戳（格式：YYYYMMDDHHMMSS，如果模拟器不存在则返回0）
 */
WtUInt64 cta_get_detail_entertime(CtxHandler cHandle, const char* stdCode, const char* openTag)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_entertime(stdCode, openTag);  // 调用模拟器的获取明细入场时间方法
}
```

#### 获取明细持仓的成本价 cta_get_detail_cost
```cpp
/**
 * @brief 获取明细持仓的成本价
 * 
 * 获取指定合约和开仓标签对应的持仓明细的成本价
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签
 * @return 成本价（如果模拟器不存在则返回0）
 */
double cta_get_detail_cost(CtxHandler cHandle, const char* stdCode, const char* openTag)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_cost(stdCode, openTag);  // 调用模拟器的获取明细成本价方法
}
```

#### 获取明细持仓的盈亏 cta_get_detail_profit
```cpp
/**
 * @brief 获取明细持仓的盈亏
 * 
 * 获取指定合约和开仓标签对应的持仓明细的盈亏
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签
 * @param flag 盈亏类型标志（0-浮动盈亏，1-平仓盈亏）
 * @return 盈亏金额（如果模拟器不存在则返回0）
 */
double cta_get_detail_profit(CtxHandler cHandle, const char* stdCode, const char* openTag, int flag)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_profit(stdCode, openTag, flag);  // 调用模拟器的获取明细盈亏方法
}
```

#### 获取所有持仓 cta_get_all_position
```cpp
/**
 * @brief 获取所有持仓
 * 
 * 枚举策略的所有持仓，通过回调函数返回每个持仓信息
 * 最后会调用一次回调函数，传入空字符串和isLast=true，表示枚举结束
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param cb 回调函数，用于接收持仓信息
 */
void cta_get_all_position(CtxHandler cHandle, FuncGetPositionCallback cb)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在
	{
		cb(cHandle, "", 0, true);  // 调用回调函数，传入空字符串表示无持仓，isLast=true表示结束
		return;
	}

	ctx->enum_position([cb, cHandle](const char* stdCode, double qty) {  // 使用lambda表达式枚举持仓
		cb(cHandle, stdCode, qty, false);  // 对每个持仓调用回调函数，isLast=false表示还有更多持仓
	}, false);  // false表示不包含未成交持仓

	cb(cHandle, "", 0, true);  // 最后调用一次回调函数，isLast=true表示枚举结束
}
```

### 价格与资金查询接口

#### 获取当前价格 cta_get_price
```cpp
/**
 * @brief 获取当前价格
 * 
 * 获取指定合约的最新价格（从回放器中获取）
 * 
 * @param stdCode 标准合约代码
 * @return 当前价格（如果没有行情数据则返回0）
 */
double cta_get_price(const char* stdCode)
{
	return getRunner().replayer().get_cur_price(stdCode);  // 通过回放器获取当前价格
}
```

#### 获取日线价格数据 cta_get_day_price
```cpp
/**
 * @brief 获取日线价格数据
 * 
 * 获取指定合约的日线价格数据（开盘价、最高价、最低价、收盘价等）
 * 
 * @param stdCode 标准合约代码
 * @param flag 价格类型标志（0-开盘价，1-最高价，2-最低价，3-收盘价）
 * @return 价格值
 */
double cta_get_day_price(const char* stdCode, int flag)
{
	return getRunner().replayer().get_day_price(stdCode, flag);  // 通过回放器获取日线价格
}
```

#### 获取资金数据 cta_get_fund_data
```cpp
/**
 * @brief 获取资金数据
 * 
 * 获取策略的资金数据（总资产、可用资金、持仓盈亏等）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param flag 资金类型标志（0-总资产，1-可用资金，2-持仓盈亏等）
 * @return 资金数值（如果模拟器不存在则返回0）
 */
double cta_get_fund_data(CtxHandler cHandle, int flag)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_fund_data(flag);  // 调用模拟器的获取资金数据方法
}
```

#### 获取交易日 cta_get_tdate
```cpp
/**
 * @brief 获取交易日
 * 
 * 获取当前交易日（格式：YYYYMMDD）
 * 
 * @return 交易日
 */
WtUInt32 cta_get_tdate()
{
	return getRunner().replayer().get_trading_date();  // 通过回放器获取交易日
}
```

#### 获取当前日期 cta_get_date
```cpp
/**
 * @brief 获取当前日期
 * 
 * 获取当前日期（格式：YYYYMMDD）
 * 
 * @return 当前日期
 */
WtUInt32 cta_get_date()
{
	return getRunner().replayer().get_date();  // 通过回放器获取当前日期
}
```

#### 获取当前时间 cta_get_time
```cpp
/**
 * @brief 获取当前时间
 * 
 * 获取当前时间（格式：HHMMSS）
 * 
 * @return 当前时间
 */
WtUInt32 cta_get_time()
{
	return getRunner().replayer().get_min_time();  // 通过回放器获取当前时间（分钟级）
}
```

### 历史数据获取接口

#### 获取K线数据 cta_get_bars
```cpp
/**
 * @brief 获取K线数据
 * 
 * 获取指定合约和周期的K线数据，通过回调函数返回
 * K线数据可能被分成多个数据块，每个数据块通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 * @param barCnt 需要获取的K线数量
 * @param isMain true表示主K线（用于策略计算），false表示辅助K线（仅用于查询）
 * @param cb 回调函数，用于接收K线数据
 * @return 实际返回的K线数量（如果模拟器不存在或发生异常则返回0）
 */
WtUInt32 cta_get_bars(CtxHandler cHandle, const char* stdCode, const char* period, WtUInt32 barCnt, bool isMain, FuncGetBarsCallback cb)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSKlineSlice* kData = ctx->stra_get_bars(stdCode, period, barCnt, isMain);  // 从模拟器获取K线数据切片
		if (kData)  // 如果数据存在
		{
			WtUInt32 reaCnt = (WtUInt32)kData->size();  // 获取实际K线数量

			for (uint32_t i = 0; i < kData->get_block_counts(); i++)  // 遍历所有数据块
				cb(cHandle, stdCode, period, kData->get_block_addr(i), kData->get_block_size(i), i == kData->get_block_counts()-1);  // 通过回调函数返回数据块

			kData->release();  // 释放K线数据切片资源
			return reaCnt;  // 返回实际K线数量
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch(...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取Tick数据 cta_get_ticks
```cpp
/**
 * @brief 获取Tick数据
 * 
 * 获取指定合约的Tick数据，通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param tickCnt 需要获取的Tick数量
 * @param cb 回调函数，用于接收Tick数据
 * @return 实际返回的Tick数量（如果模拟器不存在或发生异常则返回0）
 */
WtUInt32	cta_get_ticks(CtxHandler cHandle, const char* stdCode, WtUInt32 tickCnt, FuncGetTicksCallback cb)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSTickSlice* tData = ctx->stra_get_ticks(stdCode, tickCnt);  // 从模拟器获取Tick数据切片
		if (tData)  // 如果数据存在
		{
			uint32_t thisCnt = min(tickCnt, (WtUInt32)tData->size());  // 计算实际返回的Tick数量（取请求数量和实际数量的较小值）
			if (thisCnt != 0)  // 如果Tick数量不为0
				cb(cHandle, stdCode, (WTSTickStruct*)tData->at(0), thisCnt, true);  // 通过回调函数返回Tick数据
			else  // 如果Tick数量为0
				cb(cHandle, stdCode, NULL, 0, true);  // 通过回调函数返回空数据（表示无数据）

			tData->release();  // 释放Tick数据切片资源
			return thisCnt;
		}
		else
		{
			return 0;
		}
	}
	catch (...)
	{
		return 0;
	}
}
```

### 持仓时间与价格查询接口

#### 获取首次入场时间 cta_get_first_entertime
```cpp
/**
 * @brief 获取首次入场时间
 * 
 * 获取指定合约的首次入场时间
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 首次入场时间戳（格式：YYYYMMDDHHMMSS，0表示无持仓，如果模拟器不存在则返回0）
 */
WtUInt64 cta_get_first_entertime(CtxHandler cHandle, const char* stdCode)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_first_entertime(stdCode);  // 调用模拟器的获取首次入场时间方法
}
```

#### 获取最后入场时间 cta_get_last_entertime
```cpp
/**
 * @brief 获取最后入场时间
 * 
 * 获取指定合约的最后入场时间
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 最后入场时间戳（格式：YYYYMMDDHHMMSS，0表示无持仓，如果模拟器不存在则返回0）
 */
WtUInt64 cta_get_last_entertime(CtxHandler cHandle, const char* stdCode)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_last_entertime(stdCode);  // 调用模拟器的获取最后入场时间方法
}
```

#### 获取最后出场时间 cta_get_last_exittime
```cpp
/**
 * @brief 获取最后出场时间
 * 
 * 获取指定合约的最后出场时间
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 最后出场时间戳（格式：YYYYMMDDHHMMSS，0表示从未出场，如果模拟器不存在则返回0）
 */
WtUInt64 cta_get_last_exittime(CtxHandler cHandle, const char* stdCode)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_last_exittime(stdCode);  // 调用模拟器的获取最后出场时间方法
}
```

#### 获取最后入场价格 cta_get_last_enterprice
```cpp
/**
 * @brief 获取最后入场价格
 * 
 * 获取指定合约的最后入场价格
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 最后入场价格（0表示无持仓，如果模拟器不存在则返回0）
 */
double cta_get_last_enterprice(CtxHandler cHandle, const char* stdCode)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_last_enterprice(stdCode);  // 调用模拟器的获取最后入场价格方法
}
```

#### 获取最后入场标签 cta_get_last_entertag
```cpp
/**
 * @brief 获取最后入场标签
 * 
 * 获取指定合约的最后入场标签
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 最后入场标签字符串（空字符串表示无持仓，如果模拟器不存在则返回NULL）
 */
WtString cta_get_last_entertag(CtxHandler cHandle, const char* stdCode)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回NULL
		return 0;

	return ctx->stra_get_last_entertag(stdCode);  // 调用模拟器的获取最后入场标签方法
}
```

### 订阅接口

#### 订阅Tick行情 cta_sub_ticks
```cpp
/**
 * @brief 订阅Tick行情
 * 
 * 订阅指定合约的Tick行情，订阅后会在on_tick回调中收到该合约的Tick数据
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 */
void cta_sub_ticks(CtxHandler cHandle, const char* stdCode)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return ;

	ctx->stra_sub_ticks(stdCode);  // 调用模拟器的订阅Tick方法
}
```

#### 订阅K线事件 cta_sub_bar_events
```cpp
/**
 * @brief 订阅K线事件
 * 
 * 订阅指定合约和周期的K线闭合事件，订阅后会在on_bar回调中收到该K线的闭合事件
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 */
void cta_sub_bar_events(CtxHandler cHandle, const char* stdCode, const char* period)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->stra_sub_bar_events(stdCode, period);  // 调用模拟器的订阅K线事件方法
}
```

### 日志与数据持久化接口

#### 记录日志 cta_log_text
```cpp
/**
 * @brief 记录日志
 * 
 * 在策略上下文中记录一条日志，根据日志级别调用不同的日志方法
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param level 日志级别（LOG_LEVEL_DEBUG、LOG_LEVEL_INFO、LOG_LEVEL_WARN、LOG_LEVEL_ERROR）
 * @param message 日志消息内容
 */
void cta_log_text(CtxHandler cHandle, WtUInt32 level, const char* message)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	switch(level)  // 根据日志级别选择对应的日志方法
	{
	case LOG_LEVEL_DEBUG:  // 调试级别
		ctx->stra_log_debug(message);  // 记录调试日志
		break;
	case LOG_LEVEL_INFO:  // 信息级别
		ctx->stra_log_info(message);  // 记录信息日志
		break;
	case LOG_LEVEL_WARN:  // 警告级别
		ctx->stra_log_warn(message);  // 记录警告日志
		break;
	case LOG_LEVEL_ERROR:  // 错误级别
		ctx->stra_log_error(message);  // 记录错误日志
		break;
	default:  // 其他级别
		break;
	}
}
```

#### 保存用户数据 cta_save_userdata
```cpp
/**
 * @brief 保存用户数据
 * 
 * 保存策略的用户自定义数据（持久化存储，程序重启后仍可读取）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param key 数据键名
 * @param val 数据值（字符串格式）
 */
void cta_save_userdata(CtxHandler cHandle, const char* key, const char* val)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->stra_save_user_data(key, val);  // 调用模拟器的保存用户数据方法
}
```

#### 加载用户数据 cta_load_userdata
```cpp
/**
 * @brief 加载用户数据
 * 
 * 加载策略的用户自定义数据
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param key 数据键名
 * @param defVal 默认值（如果数据不存在则返回此值）
 * @return 数据值字符串（如果模拟器不存在则返回默认值）
 */
WtString cta_load_userdata(CtxHandler cHandle, const char* key, const char* defVal)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，返回默认值
		return defVal;

	return ctx->stra_load_user_data(key, defVal);  // 调用模拟器的加载用户数据方法
}
```

### 图表与指标接口

#### 设置图表K线 cta_set_chart_kline
```cpp
/**
 * @brief 设置图表K线
 * 
 * 为策略图表设置主K线，用于可视化展示
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 */
void cta_set_chart_kline(CtxHandler cHandle, const char* stdCode, const char* period)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->set_chart_kline(stdCode, period);  // 调用模拟器的设置图表K线方法
}
```

#### 添加图表标记 cta_add_chart_mark
```cpp
/**
 * @brief 添加图表标记
 * 
 * 在策略图表上添加一个标记点（如买卖信号）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param price 标记点的价格位置
 * @param icon 图标类型（如"buy"、"sell"等）
 * @param tag 标记标签文本
 */
void cta_add_chart_mark(CtxHandler cHandle, double price, const char* icon, const char* tag)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->add_chart_mark(price, icon, tag);  // 调用模拟器的添加图表标记方法
}
```

#### 注册指标 cta_register_index
```cpp
/**
 * @brief 注册指标
 * 
 * 在策略图表上注册一个自定义指标
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param idxName 指标名称（唯一标识）
 * @param indexType 指标类型：0-主图指标（叠加在K线上），1-副图指标（独立显示）
 */
void cta_register_index(CtxHandler cHandle, const char* idxName, WtUInt32 indexType)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->register_index(idxName, indexType);  // 调用模拟器的注册指标方法
}
```

#### 注册指标线 cta_register_index_line
```cpp
/**
 * @brief 注册指标线
 * 
 * 为已注册的指标添加一条数据线
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param idxName 指标名称
 * @param lineName 线条名称（唯一标识该线条）
 * @param lineType 线条类型：0-曲线，其他值可扩展
 * @return 是否注册成功（如果模拟器不存在则返回false）
 */
bool cta_register_index_line(CtxHandler cHandle, const char* idxName, const char* lineName, WtUInt32 lineType)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回false
		return false;

	return ctx->register_index_line(idxName, lineName, lineType);  // 调用模拟器的注册指标线方法
}
```

#### 添加指标基准线 cta_add_index_baseline
```cpp
/**
 * @brief 添加指标基准线
 * 
 * 为指标添加一条基准线（如0轴、100轴等）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param idxName 指标名称
 * @param lineName 线条名称
 * @param val 基准线数值
 * @return 是否添加成功（如果模拟器不存在则返回false）
 */
bool cta_add_index_baseline(CtxHandler cHandle, const char* idxName, const char* lineName, double val)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回false
		return false;

	return ctx->add_index_baseline(idxName, lineName, val);  // 调用模拟器的添加指标基准线方法
}
```

#### 设置指标值 cta_set_index_value
```cpp
/**
 * @brief 设置指标值
 * 
 * 更新指标线的当前值（在K线闭合时调用）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param idxName 指标名称
 * @param lineName 线条名称
 * @param val 指标值
 * @return 是否设置成功（如果模拟器不存在则返回false）
 */
bool cta_set_index_value(CtxHandler cHandle, const char* idxName, const char* lineName, double val)
{
	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回false
		return false;

	return ctx->set_index_value(idxName, lineName, val);  // 调用模拟器的设置指标值方法
}
```

### 调试接口

#### 执行策略单步回测 cta_step
```cpp
/**
 * @brief 执行策略单步回测
 * 
 * 在异步回测模式下，手动触发策略的单步执行（用于调试或精确控制回测进度）
 * 注意：只有异步模式才有意义，同步模式下直接返回false
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @return 是否执行成功（同步模式下返回false）
 */
bool cta_step(CtxHandler cHandle)
{
	//只有异步模式才有意义
	if (!getRunner().isAsync())  // 如果不是异步模式，直接返回false
		return false;

	CtaMocker* ctx = getRunner().cta_mocker();  // 获取CTA模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回false
		return false;

	return ctx->step_calc();  // 调用模拟器的单步计算方法
}
```

## SEL策略接口

### 交易操作接口

#### 设置目标持仓 sel_set_position
```cpp
```

### 持仓查询接口

#### 获取持仓数量 sel_get_position
```cpp
/**
 * @brief 设置目标持仓
 * 
 * 设置指定合约的目标持仓数量，系统会自动计算需要开仓或平仓的数量并执行
 * 注意：多因子引擎中，限价和止损价格都无效，系统会使用市价执行
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param qty 目标持仓数量（正数表示多头，负数表示空头，0表示平仓）
 * @param userTag 用户标签
 */
void sel_set_position(CtxHandler cHandle, const char* stdCode, double qty, const char* userTag)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	//多因子引擎,限价和止价都无效
	ctx->stra_set_position(stdCode, qty, userTag);  // 调用模拟器的设置目标持仓方法（多因子引擎不支持限价和止损）
}
```

#### 获取持仓盈亏 sel_get_position_profit
```cpp
/**
 * @brief 获取持仓盈亏
 * 
 * 获取指定合约的持仓浮动盈亏（扩展接口，与CTA接口同步）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 持仓盈亏金额（正数表示盈利，负数表示亏损，如果模拟器不存在则返回0）
 */
double sel_get_position_profit(CtxHandler cHandle, const char* stdCode)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_position_profit(stdCode);  // 调用模拟器的获取持仓盈亏方法
}
```

#### 获取持仓均价 sel_get_position_avgpx
```cpp
/**
 * @brief 获取持仓均价
 * 
 * 获取指定合约的持仓平均价格（扩展接口）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 持仓均价（如果模拟器不存在则返回0）
 */
double sel_get_position_avgpx(CtxHandler cHandle, const char* stdCode)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_position_avgpx(stdCode);  // 调用模拟器的获取持仓均价方法
}
```

#### 获取明细持仓的入场时间 sel_get_detail_entertime
```cpp
/**
 * @brief 获取明细持仓的入场时间
 * 
 * 获取指定合约和开仓标签对应的持仓明细的入场时间（扩展接口）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签
 * @return 入场时间戳（格式：YYYYMMDDHHMMSS，如果模拟器不存在则返回0）
 */
WtUInt64 sel_get_detail_entertime(CtxHandler cHandle, const char* stdCode, const char* openTag)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_entertime(stdCode, openTag);  // 调用模拟器的获取明细入场时间方法
}
```

#### 获取明细持仓的成本价 sel_get_detail_cost
```cpp
/**
 * @brief 获取明细持仓的成本价
 * 
 * 获取指定合约和开仓标签对应的持仓明细的成本价（扩展接口）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签
 * @return 成本价（如果模拟器不存在则返回0）
 */
double sel_get_detail_cost(CtxHandler cHandle, const char* stdCode, const char* openTag)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_cost(stdCode, openTag);  // 调用模拟器的获取明细成本价方法
}
```

#### 获取明细持仓的盈亏 sel_get_detail_profit
```cpp
/**
 * @brief 获取明细持仓的盈亏
 * 
 * 获取指定合约和开仓标签对应的持仓明细的盈亏（扩展接口）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param openTag 开仓标签
 * @param flag 盈亏类型标志（0-浮动盈亏，1-平仓盈亏）
 * @return 盈亏金额（如果模拟器不存在则返回0）
 */
double sel_get_detail_profit(CtxHandler cHandle, const char* stdCode, const char* openTag, int flag)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_detail_profit(stdCode, openTag, flag);  // 调用模拟器的获取明细盈亏方法
}
```

#### 获取所有持仓 sel_get_all_position
```cpp
/**
 * @brief 获取所有持仓
 * 
 * 枚举策略的所有持仓，通过回调函数返回每个持仓信息
 * 最后会调用一次回调函数，传入空字符串和isLast=true，表示枚举结束
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param cb 回调函数，用于接收持仓信息
 */
void sel_get_all_position(CtxHandler cHandle, FuncGetPositionCallback cb)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在
	{
		cb(cHandle, "", 0, true);  // 调用回调函数，传入空字符串表示无持仓，isLast=true表示结束
		return;
	}

	ctx->enum_position([cb, cHandle](const char* stdCode, double qty) {  // 使用lambda表达式枚举持仓
		cb(cHandle, stdCode, qty, false);  // 对每个持仓调用回调函数，isLast=false表示还有更多持仓
	});

	cb(cHandle, "", 0, true);  // 最后调用一次回调函数，isLast=true表示枚举结束
}
```

### 价格与资金查询接口

#### 获取当前价格 sel_get_price
```cpp
/**
 * @brief 获取当前价格
 * 
 * 获取指定合约的最新价格（从回放器中获取）
 * 
 * @param stdCode 标准合约代码
 * @return 当前价格（如果没有行情数据则返回0）
 */
double sel_get_price(const char* stdCode)
{
	return getRunner().replayer().get_cur_price(stdCode);  // 通过回放器获取当前价格
}
```

#### 获取日线价格数据 sel_get_day_price
```cpp
/**
 * @brief 获取日线价格数据
 * 
 * 获取指定合约的日线价格数据（开盘价、最高价、最低价、收盘价等）
 * 
 * @param stdCode 标准合约代码
 * @param flag 价格类型标志（0-开盘价，1-最高价，2-最低价，3-收盘价）
 * @return 价格值
 */
double sel_get_day_price(const char* stdCode, int flag)
{
	return getRunner().replayer().get_day_price(stdCode, flag);  // 通过回放器获取日线价格
}
```

#### 获取资金数据 sel_get_fund_data
```cpp
/**
 * @brief 获取资金数据
 * 
 * 获取策略的资金数据（总资产、可用资金、持仓盈亏等）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param flag 资金类型标志（0-总资产，1-可用资金，2-持仓盈亏等）
 * @return 资金数值（如果模拟器不存在则返回0）
 */
double sel_get_fund_data(CtxHandler cHandle, int flag)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_fund_data(flag);  // 调用模拟器的获取资金数据方法
}
```

#### 获取交易日 sel_get_tdate
```cpp
/**
 * @brief 获取交易日
 * 
 * 获取当前交易日（格式：YYYYMMDD）
 * 
 * @return 交易日
 */
WtUInt32 sel_get_tdate()
{
	return getRunner().replayer().get_trading_date();  // 通过回放器获取交易日
}
```

#### 获取当前日期 sel_get_date
```cpp
/**
 * @brief 获取当前日期
 * 
 * 获取当前日期（格式：YYYYMMDD）
 * 
 * @return 当前日期
 */
WtUInt32 sel_get_date()
{
	return getRunner().replayer().get_date();  // 通过回放器获取当前日期
}
```

#### 获取当前时间 sel_get_time
```cpp
/**
 * @brief 获取当前时间
 * 
 * 获取当前时间（格式：HHMMSS）
 * 
 * @return 当前时间
 */
WtUInt32 sel_get_time()
{
	return getRunner().replayer().get_min_time();  // 通过回放器获取当前时间（分钟级）
}
```

### 历史数据获取接口

#### 获取K线数据 sel_get_bars
```cpp
/**
 * @brief 获取K线数据
 * 
 * 获取指定合约和周期的K线数据，通过回调函数返回
 * K线数据可能被分成多个数据块，每个数据块通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 * @param barCnt 需要获取的K线数量
 * @param cb 回调函数，用于接收K线数据
 * @return 实际返回的K线数量（如果模拟器不存在或发生异常则返回0）
 */
WtUInt32 sel_get_bars(CtxHandler cHandle, const char* stdCode, const char* period, WtUInt32 barCnt, FuncGetBarsCallback cb)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSKlineSlice* kData = ctx->stra_get_bars(stdCode, period, barCnt);  // 从模拟器获取K线数据切片
		if (kData)  // 如果数据存在
		{
			WtUInt32 reaCnt = (WtUInt32)kData->size();  // 获取实际K线数量

			for (uint32_t i = 0; i < kData->get_block_counts(); i++)  // 遍历所有数据块
				cb(cHandle, stdCode, period, kData->get_block_addr(i), kData->get_block_size(i), i == kData->get_block_counts() - 1);  // 通过回调函数返回数据块

			kData->release();  // 释放K线数据切片资源
			return reaCnt;  // 返回实际K线数量
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取Tick数据 sel_get_ticks
```cpp
/**
 * @brief 获取Tick数据
 * 
 * 获取指定合约的Tick数据，通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param tickCnt 需要获取的Tick数量
 * @param cb 回调函数，用于接收Tick数据
 * @return 实际返回的Tick数量（如果模拟器不存在或发生异常则返回0）
 */
WtUInt32	sel_get_ticks(CtxHandler cHandle, const char* stdCode, WtUInt32 tickCnt, FuncGetTicksCallback cb)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSTickSlice* tData = ctx->stra_get_ticks(stdCode, tickCnt);  // 从模拟器获取Tick数据切片
		if (tData)  // 如果数据存在
		{
			uint32_t thisCnt = min(tickCnt, (WtUInt32)tData->size());  // 计算实际返回的Tick数量（取请求数量和实际数量的较小值）
			if (thisCnt != 0)  // 如果Tick数量不为0
				cb(cHandle, stdCode, (WTSTickStruct*)tData->at(0), thisCnt, true);  // 通过回调函数返回Tick数据
			else  // 如果Tick数量为0
				cb(cHandle, stdCode, NULL, 0, true);  // 通过回调函数返回空数据（表示无数据）
			tData->release();  // 释放Tick数据切片资源
			return thisCnt;  // 返回实际Tick数量
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

### 持仓时间与价格查询接口

#### 获取首次入场时间 sel_get_first_entertime
```cpp
/**
 * @brief 获取首次入场时间
 * 
 * 获取指定合约的首次入场时间（扩展接口）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 首次入场时间戳（格式：YYYYMMDDHHMMSS，0表示无持仓，如果模拟器不存在则返回0）
 */
WtUInt64 sel_get_first_entertime(CtxHandler cHandle, const char* stdCode)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_first_entertime(stdCode);  // 调用模拟器的获取首次入场时间方法
}
```

#### 获取最后入场时间 sel_get_last_entertime
```cpp
/**
 * @brief 获取最后入场时间
 * 
 * 获取指定合约的最后入场时间（扩展接口）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 最后入场时间戳（格式：YYYYMMDDHHMMSS，0表示无持仓，如果模拟器不存在则返回0）
 */
WtUInt64 sel_get_last_entertime(CtxHandler cHandle, const char* stdCode)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_last_entertime(stdCode);  // 调用模拟器的获取最后入场时间方法
}
```

#### 获取最后出场时间 sel_get_last_exittime
```cpp
/**
 * @brief 获取最后出场时间
 * 
 * 获取指定合约的最后出场时间（扩展接口）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 最后出场时间戳（格式：YYYYMMDDHHMMSS，0表示从未出场，如果模拟器不存在则返回0）
 */
WtUInt64 sel_get_last_exittime(CtxHandler cHandle, const char* stdCode)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_last_exittime(stdCode);  // 调用模拟器的获取最后出场时间方法
}
```

#### 获取最后入场价格 sel_get_last_enterprice
```cpp
/**
 * @brief 获取最后入场价格
 * 
 * 获取指定合约的最后入场价格（扩展接口）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 最后入场价格（0表示无持仓，如果模拟器不存在则返回0）
 */
double sel_get_last_enterprice(CtxHandler cHandle, const char* stdCode)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return ctx->stra_get_last_enterprice(stdCode);  // 调用模拟器的获取最后入场价格方法
}
```

#### 获取最后入场标签 sel_get_last_entertag
```cpp
/**
 * @brief 获取最后入场标签
 * 
 * 获取指定合约的最后入场标签（扩展接口）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 最后入场标签字符串（空字符串表示无持仓，如果模拟器不存在则返回NULL）
 */
WtString sel_get_last_entertag(CtxHandler cHandle, const char* stdCode)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回NULL
		return 0;

	return ctx->stra_get_last_entertag(stdCode);  // 调用模拟器的获取最后入场标签方法
}
```

### 订阅接口

#### 订阅Tick行情 sel_sub_ticks
```cpp
/**
 * @brief 订阅Tick行情
 * 
 * 订阅指定合约的Tick行情，订阅后会在on_tick回调中收到该合约的Tick数据
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 */
void sel_sub_ticks(CtxHandler cHandle, const char* stdCode)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->stra_sub_ticks(stdCode);  // 调用模拟器的订阅Tick方法
}
```

### 日志与数据持久化接口

#### 记录日志 sel_log_text
```cpp
/**
 * @brief 记录日志
 * 
 * 在策略上下文中记录一条日志，根据日志级别调用不同的日志方法
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param level 日志级别（LOG_LEVEL_DEBUG、LOG_LEVEL_INFO、LOG_LEVEL_WARN、LOG_LEVEL_ERROR）
 * @param message 日志消息内容
 */
void sel_log_text(CtxHandler cHandle, WtUInt32 level, const char* message)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	switch (level)  // 根据日志级别选择对应的日志方法
	{
	case LOG_LEVEL_DEBUG:  // 调试级别
		ctx->stra_log_debug(message);  // 记录调试日志
		break;
	case LOG_LEVEL_INFO:  // 信息级别
		ctx->stra_log_info(message);  // 记录信息日志
		break;
	case LOG_LEVEL_WARN:  // 警告级别
		ctx->stra_log_warn(message);  // 记录警告日志
		break;
	case LOG_LEVEL_ERROR:  // 错误级别
		ctx->stra_log_error(message);  // 记录错误日志
		break;
	default:  // 其他级别
		break;
	}
}
```

#### 保存用户数据 sel_save_userdata
```cpp
/**
 * @brief 保存用户数据
 * 
 * 保存策略的用户自定义数据（持久化存储，程序重启后仍可读取）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param key 数据键名
 * @param val 数据值（字符串格式）
 */
void sel_save_userdata(CtxHandler cHandle, const char* key, const char* val)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	ctx->stra_save_user_data(key, val);  // 调用模拟器的保存用户数据方法
}
```

#### 加载用户数据 sel_load_userdata
```cpp
/**
 * @brief 加载用户数据
 * 
 * 加载策略的用户自定义数据
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param key 数据键名
 * @param defVal 默认值（如果数据不存在则返回此值）
 * @return 数据值字符串（如果模拟器不存在则返回默认值）
 */
WtString sel_load_userdata(CtxHandler cHandle, const char* key, const char* defVal)
{
	SelMocker* ctx = getRunner().sel_mocker();  // 获取SEL模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，返回默认值
		return defVal;

	return ctx->stra_load_user_data(key, defVal);  // 调用模拟器的加载用户数据方法
}
```

## HFT策略接口

### 交易操作接口

#### 买入 hft_buy
```cpp
/**
 * @brief 买入
 * 
 * 执行买入操作，提交买入订单
 * 如果订单被拆单，会返回多个订单ID（逗号分隔）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param price 买入价格（0表示市价）
 * @param qty 买入数量
 * @param userTag 用户标签（用于标识该笔交易）
 * @param flag 订单标志（0-普通单，其他值可扩展）
 * @return 订单ID列表（逗号分隔的字符串，如果拆单则返回多个订单ID，如果模拟器不存在则返回空字符串）
 */
WtString hft_buy(CtxHandler cHandle, const char* stdCode, double price, double qty, const char* userTag, int flag)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，返回空字符串
		return "";

	static std::string ret;  // 静态变量，用于存储返回的订单ID列表字符串

	std::stringstream ss;  // 字符串流，用于构建订单ID列表
	OrderIDs ids = mocker->stra_buy(stdCode, price, qty, userTag, flag);  // 调用模拟器的买入方法
	for (WtUInt32 localid : ids)  // 遍历所有订单ID（如果拆单则可能有多个）
	{
		ss << localid << ",";  // 将订单ID添加到字符串流中，用逗号分隔
	}

	ret = ss.str();  // 将字符串流转换为字符串
	ret = ret.substr(0, ret.size() - 1);  // 移除最后一个逗号
	return ret.c_str();  // 返回C风格字符串
}
```

#### 卖出 hft_sell
```cpp
/**
 * @brief 卖出
 * 
 * 执行卖出操作，提交卖出订单
 * 如果订单被拆单，会返回多个订单ID（逗号分隔）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param price 卖出价格（0表示市价）
 * @param qty 卖出数量
 * @param userTag 用户标签（用于标识该笔交易）
 * @param flag 订单标志（0-普通单，其他值可扩展）
 * @return 订单ID列表（逗号分隔的字符串，如果拆单则返回多个订单ID，如果模拟器不存在则返回空字符串）
 */
WtString hft_sell(CtxHandler cHandle, const char* stdCode, double price, double qty, const char* userTag, int flag)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，返回空字符串
		return "";

	static std::string ret;  // 静态变量，用于存储返回的订单ID列表字符串

	std::stringstream ss;  // 字符串流，用于构建订单ID列表
	OrderIDs ids = mocker->stra_sell(stdCode, price, qty, userTag, flag);  // 调用模拟器的卖出方法
	for (WtUInt32 localid : ids)  // 遍历所有订单ID（如果拆单则可能有多个）
	{
		ss << localid << ",";  // 将订单ID添加到字符串流中，用逗号分隔
	}

	ret = ss.str();  // 将字符串流转换为字符串
	ret = ret.substr(0, ret.size() - 1);  // 移除最后一个逗号
	return ret.c_str();  // 返回C风格字符串
}
```

#### 撤销订单 hft_cancel
```cpp
/**
 * @brief 撤销订单
 * 
 * 撤销指定的订单
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param localid 本地订单ID（下单时返回的订单ID）
 * @return 是否撤销成功（如果模拟器不存在则返回false）
 */
bool hft_cancel(CtxHandler cHandle, WtUInt32 localid)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回false
		return false;

	return mocker->stra_cancel(localid);  // 调用模拟器的撤销订单方法
}
```

#### 撤销所有订单 hft_cancel_all
```cpp
/**
 * @brief 撤销所有订单
 * 
 * 撤销指定合约和方向的所有未完成订单
 * 返回被撤销的订单ID列表（逗号分隔的字符串）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码（NULL或空字符串表示所有合约）
 * @param isBuy true表示撤销买入订单，false表示撤销卖出订单
 * @return 被撤销的订单ID列表（逗号分隔的字符串，如果模拟器不存在则返回空字符串）
 */
WtString hft_cancel_all(CtxHandler cHandle, const char* stdCode, bool isBuy)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，返回空字符串
		return "";

	static thread_local std::string ret;  // 线程局部静态变量，用于存储返回的订单ID列表字符串

	std::stringstream ss;  // 字符串流，用于构建订单ID列表
	OrderIDs ids = mocker->stra_cancel(stdCode, isBuy, DBL_MAX);  // 调用模拟器的撤销所有订单方法，DBL_MAX表示撤销所有价格
	for (WtUInt32 localid : ids)  // 遍历所有被撤销的订单ID
	{
		ss << localid << ",";  // 将订单ID添加到字符串流中，用逗号分隔
	}

	ret = ss.str();  // 将字符串流转换为字符串
	if (ret.size() > 0)  // 如果字符串不为空
		ret = ret.substr(0, ret.size() - 1);  // 移除最后一个逗号
	return ret.c_str();  // 返回C风格字符串
}
```

### 持仓查询接口

#### 获取持仓数量 hft_get_position
```cpp
/**
 * @brief 获取持仓数量
 * 
 * 获取指定合约的持仓数量
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param bOnlyValid true表示只返回有效持仓（已成交的），false表示返回所有持仓（包括未成交的）
 * @return 持仓数量（正数表示多头，负数表示空头，如果模拟器不存在则返回0）
 */
double hft_get_position(CtxHandler cHandle, const char* stdCode, bool bOnlyValid)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return mocker->stra_get_position(stdCode, bOnlyValid);  // 调用模拟器的获取持仓方法
}
```

#### 获取持仓盈亏 hft_get_position_profit
```cpp
/**
 * @brief 获取持仓盈亏
 * 
 * 获取指定合约的持仓浮动盈亏
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 持仓盈亏金额（正数表示盈利，负数表示亏损，如果模拟器不存在则返回0）
 */
double hft_get_position_profit(CtxHandler cHandle, const char* stdCode)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return mocker->stra_get_position_profit(stdCode);  // 调用模拟器的获取持仓盈亏方法
}
```

#### 获取持仓均价 hft_get_position_avgpx
```cpp
/**
 * @brief 获取持仓均价
 * 
 * 获取指定合约的持仓平均价格
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 持仓均价（如果模拟器不存在则返回0）
 */
double hft_get_position_avgpx(CtxHandler cHandle, const char* stdCode)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return mocker->stra_get_position_avgpx(stdCode);  // 调用模拟器的获取持仓均价方法
}
```

#### 获取未完成订单数量 hft_get_undone
```cpp
/**
 * @brief 获取未完成订单数量
 * 
 * 获取指定合约的未完成订单数量（包括未成交和部分成交的订单）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @return 未完成订单数量（正数表示买入未完成，负数表示卖出未完成，如果模拟器不存在则返回0）
 */
double hft_get_undone(CtxHandler cHandle, const char* stdCode)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	return mocker->stra_get_undone(stdCode);  // 调用模拟器的获取未完成订单数量方法
}
```

### 价格与时间查询接口

#### 获取当前价格 hft_get_price
```cpp
/**
 * @brief 获取当前价格
 * 
 * 获取指定合约的最新价格（从回放器中获取）
 * 
 * @param stdCode 标准合约代码
 * @return 当前价格（如果没有行情数据则返回0）
 */
double hft_get_price(const char* stdCode)
{
	return getRunner().replayer().get_cur_price(stdCode);  // 通过回放器获取当前价格
}
```

#### 获取当前日期 hft_get_date
```cpp
/**
 * @brief 获取当前日期
 * 
 * 获取当前日期（格式：YYYYMMDD）
 * 
 * @return 当前日期
 */
WtUInt32 hft_get_date()
{
	return getRunner().replayer().get_date();  // 通过回放器获取当前日期
}
```

#### 获取当前时间 hft_get_time
```cpp
/**
 * @brief 获取当前时间
 * 
 * 获取当前时间（格式：HHMMSS，原始时间，包含毫秒信息）
 * 
 * @return 当前时间
 */
WtUInt32 hft_get_time()
{
	return getRunner().replayer().get_raw_time();  // 通过回放器获取当前原始时间（包含毫秒）
}
```

#### 获取当前秒数 hft_get_secs
```cpp
/**
 * @brief 获取当前秒数
 * 
 * 获取当前时间的秒数部分（0-59）
 * 
 * @return 当前秒数
 */
WtUInt32 hft_get_secs()
{
	return getRunner().replayer().get_secs();  // 通过回放器获取当前秒数
}
```

### 历史数据获取接口

#### 获取K线数据 hft_get_bars
```cpp
/**
 * @brief 获取K线数据
 * 
 * 获取指定合约和周期的K线数据，通过回调函数返回
 * K线数据可能被分成多个数据块，每个数据块通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 * @param barCnt 需要获取的K线数量
 * @param cb 回调函数，用于接收K线数据
 * @return 实际返回的K线数量（如果模拟器不存在或发生异常则返回0）
 */
WtUInt32 hft_get_bars(CtxHandler cHandle, const char* stdCode, const char* period, WtUInt32 barCnt, FuncGetBarsCallback cb)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回0
		return 0;

	try  // 使用try-catch捕获可能的异常
	{
		WTSKlineSlice* kData = mocker->stra_get_bars(stdCode, period, barCnt);  // 从模拟器获取K线数据切片
		if (kData)  // 如果数据存在
		{
			WtUInt32 reaCnt = (WtUInt32)kData->size();  // 获取实际K线数量

			for (uint32_t i = 0; i < kData->get_block_counts(); i++)  // 遍历所有数据块
				cb(cHandle, stdCode, period, kData->get_block_addr(i), kData->get_block_size(i), i == kData->get_block_counts() - 1);  // 通过回调函数返回数据块

			kData->release();  // 释放K线数据切片资源
			return reaCnt;  // 返回实际K线数量
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取Tick数据 hft_get_ticks
```cpp
/**
 * @brief 获取Tick数据
 * 
 * 获取指定合约的Tick数据，通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param tickCnt 需要获取的Tick数量
 * @param cb 回调函数，用于接收Tick数据
 * @return 实际返回的Tick数量（如果模拟器不存在或发生异常则返回0）
 */
WtUInt32 hft_get_ticks(CtxHandler cHandle, const char* stdCode, WtUInt32 tickCnt, FuncGetTicksCallback cb)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSTickSlice* tData = mocker->stra_get_ticks(stdCode, tickCnt);  // 从模拟器获取Tick数据切片
		if (tData)  // 如果数据存在
		{
			uint32_t thisCnt = min(tickCnt, (WtUInt32)tData->size());  // 计算实际返回的Tick数量（取请求数量和实际数量的较小值）
			if(thisCnt != 0)  // 如果Tick数量不为0
				cb(cHandle, stdCode, (WTSTickStruct*)tData->at(0), thisCnt, true);  // 通过回调函数返回Tick数据
			else  // 如果Tick数量为0
				cb(cHandle, stdCode, NULL, 0, true);  // 通过回调函数返回空数据（表示无数据）
			tData->release();  // 释放Tick数据切片资源
			return thisCnt;  // 返回实际Tick数量
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取订单队列数据 hft_get_ordque
```cpp
/**
 * @brief 获取订单队列数据
 * 
 * 获取指定合约的订单队列数据（Level2行情），通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param itemCnt 需要获取的数据条数
 * @param cb 回调函数，用于接收订单队列数据
 * @return 实际返回的数据条数（如果模拟器不存在或发生异常则返回0）
 */
WtUInt32 hft_get_ordque(CtxHandler cHandle, const char* stdCode, WtUInt32 itemCnt, FuncGetOrdQueCallback cb)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSOrdQueSlice* dataSlice = mocker->stra_get_order_queue(stdCode, itemCnt);  // 从模拟器获取订单队列数据切片
		if (dataSlice)  // 如果数据存在
		{
			uint32_t thisCnt = min(itemCnt, (WtUInt32)dataSlice->size());  // 计算实际返回的数据条数（取请求数量和实际数量的较小值）
			cb(cHandle, stdCode, (WTSOrdQueStruct*)dataSlice->at(0), thisCnt, true);  // 通过回调函数返回订单队列数据
			dataSlice->release();  // 释放订单队列数据切片资源
			return thisCnt;  // 返回实际数据条数
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取订单明细数据 hft_get_orddtl
```cpp
/**
 * @brief 获取订单明细数据
 * 
 * 获取指定合约的订单明细数据（Level2行情），通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param itemCnt 需要获取的数据条数
 * @param cb 回调函数，用于接收订单明细数据
 * @return 实际返回的数据条数（如果模拟器不存在或发生异常则返回0）
 */
WtUInt32 hft_get_orddtl(CtxHandler cHandle, const char* stdCode, WtUInt32 itemCnt, FuncGetOrdDtlCallback cb)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSOrdDtlSlice* dataSlice = mocker->stra_get_order_detail(stdCode, itemCnt);  // 从模拟器获取订单明细数据切片
		if (dataSlice)  // 如果数据存在
		{
			uint32_t thisCnt = min(itemCnt, (WtUInt32)dataSlice->size());  // 计算实际返回的数据条数（取请求数量和实际数量的较小值）
			cb(cHandle, stdCode, (WTSOrdDtlStruct*)dataSlice->at(0), thisCnt, true);  // 通过回调函数返回订单明细数据
			dataSlice->release();  // 释放订单明细数据切片资源
			return thisCnt;  // 返回实际数据条数
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

#### 获取逐笔成交数据 hft_get_trans
```cpp
/**
 * @brief 获取逐笔成交数据
 * 
 * 获取指定合约的逐笔成交数据（Level2行情），通过回调函数返回
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 * @param itemCnt 需要获取的数据条数
 * @param cb 回调函数，用于接收逐笔成交数据
 * @return 实际返回的数据条数（如果模拟器不存在或发生异常则返回0）
 */
WtUInt32 hft_get_trans(CtxHandler cHandle, const char* stdCode, WtUInt32 itemCnt, FuncGetTransCallback cb)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回0
		return 0;
	try  // 使用try-catch捕获可能的异常
	{
		WTSTransSlice* dataSlice = mocker->stra_get_transaction(stdCode, itemCnt);  // 从模拟器获取逐笔成交数据切片
		if (dataSlice)  // 如果数据存在
		{
			uint32_t thisCnt = min(itemCnt, (WtUInt32)dataSlice->size());  // 计算实际返回的数据条数（取请求数量和实际数量的较小值）
			cb(cHandle, stdCode, (WTSTransStruct*)dataSlice->at(0), thisCnt, true);  // 通过回调函数返回逐笔成交数据
			dataSlice->release();  // 释放逐笔成交数据切片资源
			return thisCnt;  // 返回实际数据条数
		}
		else  // 如果数据不存在
		{
			return 0;  // 返回0
		}
	}
	catch (...)  // 捕获所有异常
	{
		return 0;  // 发生异常时返回0
	}
}
```

### 订阅接口

#### 订阅Tick行情 hft_sub_ticks
```cpp
/**
 * @brief 订阅Tick行情
 * 
 * 订阅指定合约的Tick行情，订阅后会在on_tick回调中收到该合约的Tick数据
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 */
void hft_sub_ticks(CtxHandler cHandle, const char* stdCode)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回
		return;

	mocker->stra_sub_ticks(stdCode);  // 调用模拟器的订阅Tick方法
}
```

#### 订阅订单队列 hft_sub_order_queue
```cpp
/**
 * @brief 订阅订单队列
 * 
 * 订阅指定合约的订单队列数据，订阅后会在on_order_queue回调中收到该合约的订单队列数据
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 */
void hft_sub_order_queue(CtxHandler cHandle, const char* stdCode)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回
		return;

	mocker->stra_sub_order_queues(stdCode);  // 调用模拟器的订阅订单队列方法
}
```

#### 订阅订单明细 hft_sub_order_detail
```cpp
/**
 * @brief 订阅订单明细
 * 
 * 订阅指定合约的订单明细数据，订阅后会在on_order_detail回调中收到该合约的订单明细数据
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 */
void hft_sub_order_detail(CtxHandler cHandle, const char* stdCode)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回
		return;

	mocker->stra_sub_order_details(stdCode);  // 调用模拟器的订阅订单明细方法
}
```

#### 订阅逐笔成交 hft_sub_transaction
```cpp
/**
 * @brief 订阅逐笔成交
 * 
 * 订阅指定合约的逐笔成交数据，订阅后会在on_transaction回调中收到该合约的逐笔成交数据
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param stdCode 标准合约代码
 */
void hft_sub_transaction(CtxHandler cHandle, const char* stdCode)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回
		return;

	mocker->stra_sub_transactions(stdCode);  // 调用模拟器的订阅逐笔成交方法
}
```

### 日志与数据持久化接口

#### 记录日志 hft_log_text
```cpp
/**
 * @brief 记录日志
 * 
 * 在策略上下文中记录一条日志，根据日志级别调用不同的日志方法
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param level 日志级别（LOG_LEVEL_DEBUG、LOG_LEVEL_INFO、LOG_LEVEL_WARN、LOG_LEVEL_ERROR）
 * @param message 日志消息内容
 */
void hft_log_text(CtxHandler cHandle, WtUInt32 level, const char* message)
{
	HftMocker* ctx = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (ctx == NULL)  // 如果模拟器不存在，直接返回
		return;

	switch (level)  // 根据日志级别选择对应的日志方法
	{
	case LOG_LEVEL_DEBUG:  // 调试级别
		ctx->stra_log_debug(message);  // 记录调试日志
		break;
	case LOG_LEVEL_INFO:  // 信息级别
		ctx->stra_log_info(message);  // 记录信息日志
		break;
	case LOG_LEVEL_WARN:  // 警告级别
		ctx->stra_log_warn(message);  // 记录警告日志
		break;
	case LOG_LEVEL_ERROR:  // 错误级别
		ctx->stra_log_error(message);  // 记录错误日志
		break;
	default:  // 其他级别
		break;
	}
}
```

#### 保存用户数据 hft_save_userdata
```cpp
/**
 * @brief 保存用户数据
 * 
 * 保存策略的用户自定义数据（持久化存储，程序重启后仍可读取）
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param key 数据键名
 * @param val 数据值（字符串格式）
 */
void hft_save_userdata(CtxHandler cHandle, const char* key, const char* val)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回
		return;

	mocker->stra_save_user_data(key, val);  // 调用模拟器的保存用户数据方法
}
```

#### 加载用户数据 hft_load_userdata
```cpp
/**
 * @brief 加载用户数据
 * 
 * 加载策略的用户自定义数据
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 * @param key 数据键名
 * @param defVal 默认值（如果数据不存在则返回此值）
 * @return 数据值字符串（如果模拟器不存在则返回默认值）
 */
WtString hft_load_userdata(CtxHandler cHandle, const char* key, const char* defVal)
{
	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，返回默认值
		return defVal;

	return mocker->stra_load_user_data(key, defVal);  // 调用模拟器的加载用户数据方法
}
```

### 调试接口

#### 执行策略单步回测 hft_step
```cpp
/**
 * @brief 执行策略单步回测
 * 
 * 在异步回测模式下，手动触发策略的单步执行（用于调试或精确控制回测进度）
 * 注意：只有异步模式才有意义，同步模式下直接返回
 * 
 * @param cHandle 策略上下文句柄（回测模式下此参数未使用）
 */
void hft_step(CtxHandler cHandle)
{
	//只有异步模式才有意义
	if (!getRunner().isAsync())  // 如果不是异步模式，直接返回
		return;

	HftMocker* mocker = getRunner().hft_mocker();  // 获取HFT模拟器对象
	if (mocker == NULL)  // 如果模拟器不存在，直接返回
		return;

	mocker->step_tick();  // 调用模拟器的单步Tick处理方法
}
```

# 回测运行器 WtBtRunner.h/cpp

## 成员
- **CTA策略回调函数**
  - `FuncStraInitCallback _cb_cta_init`：CTA策略初始化回调函数
  - `FuncStraTickCallback _cb_cta_tick`：CTA策略Tick更新回调函数
  - `FuncStraCalcCallback _cb_cta_calc`：CTA策略计算回调函数
  - `FuncStraCalcCallback _cb_cta_calc_done`：CTA策略计算完成回调函数
  - `FuncStraBarCallback _cb_cta_bar`：CTA策略K线闭合回调函数
  - `FuncSessionEvtCallback _cb_cta_sessevt`：CTA策略交易日事件回调函数
  - `FuncStraCondTriggerCallback _cb_cta_cond_trigger`：CTA策略条件单触发回调函数

- **SEL策略回调函数**
  - `FuncStraInitCallback _cb_sel_init`：SEL策略初始化回调函数
  - `FuncStraTickCallback _cb_sel_tick`：SEL策略Tick更新回调函数
  - `FuncStraCalcCallback _cb_sel_calc`：SEL策略计算回调函数
  - `FuncStraCalcCallback _cb_sel_calc_done`：SEL策略计算完成回调函数
  - `FuncStraBarCallback _cb_sel_bar`：SEL策略K线闭合回调函数
  - `FuncSessionEvtCallback _cb_sel_sessevt`：SEL策略交易日事件回调函数

- **HFT策略回调函数**
  - `FuncStraInitCallback _cb_hft_init`：HFT策略初始化回调函数
  - `FuncStraTickCallback _cb_hft_tick`：HFT策略Tick更新回调函数
  - `FuncStraBarCallback _cb_hft_bar`：HFT策略K线闭合回调函数
  - `FuncSessionEvtCallback _cb_hft_sessevt`：HFT策略交易日事件回调函数
  - `FuncHftChannelCallback _cb_hft_chnl`：HFT策略交易通道事件回调函数
  - `FuncHftOrdCallback _cb_hft_ord`：HFT策略订单状态变化回调函数
  - `FuncHftTrdCallback _cb_hft_trd`：HFT策略成交回报回调函数
  - `FuncHftEntrustCallback _cb_hft_entrust`：HFT策略委托回报回调函数
  - `FuncStraOrdQueCallback _cb_hft_ordque`：HFT策略订单队列更新回调函数
  - `FuncStraOrdDtlCallback _cb_hft_orddtl`：HFT策略订单明细更新回调函数
  - `FuncStraTransCallback _cb_hft_trans`：HFT策略逐笔成交更新回调函数

- **引擎事件回调函数**
  - `FuncEventCallback _cb_evt`：引擎事件回调函数（引擎初始化、调度、交易日、回测结束等事件）

- **外部数据加载器回调函数**
  - `FuncLoadFnlBars _ext_fnl_bar_loader`：最终K线数据加载器回调函数（已复权）
  - `FuncLoadRawBars _ext_raw_bar_loader`：原始K线数据加载器回调函数（未复权）
  - `FuncLoadAdjFactors _ext_adj_fct_loader`：复权因子加载器回调函数
  - `FuncLoadRawTicks _ext_tick_loader`：Tick数据加载器回调函数
  - `bool _loader_auto_trans`：是否自动转储数据到本地缓存

- **策略模拟器对象指针**
  - `CtaMocker* _cta_mocker`：CTA策略模拟器对象指针
  - `SelMocker* _sel_mocker`：SEL策略模拟器对象指针
  - `HftMocker* _hft_mocker`：HFT策略模拟器对象指针
  - `ExecMocker* _exec_mocker`：执行器模拟器对象指针

- **核心组件对象**
  - `HisDataReplayer _replayer`：历史数据回放器对象（负责回放历史数据，驱动策略执行）
  - `EventNotifier _notifier`：事件通知器对象（负责管理事件通知）

- **状态标志**
  - `bool _inited`：是否已初始化
  - `bool _running`：是否正在运行

- **异步回测相关**
  - `StdThreadPtr _worker`：工作线程指针（异步模式下使用）
  - `bool _async`：是否异步模式

- **数据推送相关**
  - `void* _feed_obj`：数据推送用户对象指针（传递给数据加载回调函数）
  - `FuncReadBars _feeder_bars`：K线数据推送回调函数（用于接收外部加载的K线数据）
  - `FuncReadTicks _feeder_ticks`：Tick数据推送回调函数（用于接收外部加载的Tick数据）
  - `FuncReadFactors _feeder_fcts`：复权因子推送回调函数（用于接收外部加载的复权因子数据）
  - `StdUniqueMutex _feed_mtx`：数据推送互斥锁（保证数据加载过程的线程安全）

- **配置对象**
  - `WTSVariant* _cfg`：配置对象指针（存储回测引擎的配置信息）

## IBtDataLoader 接口实现

### 加载最终K线数据 loadFinalHisBars
当 C++ 内部的数据管理器（`WtDataMgr`）发现本地缓存或存储中缺失所需的**历史 K 线数据（特指复权后的数据）**时，它会通过此函数调用上层（如 Python 端）注册的加载器函数，从外部数据源（如 SQL 数据库、CSV 文件、第三方 API）按需加载数据。
* **调用方**：C++ 核心层的 `WtDataMgr`（数据管理器）。
* **执行方**：外部语言（Python/C#）注册的 `FuncLoadFnlBars` 回调函数。
* **数据类型**：**复权 K 线数据**（Final Bars），即经过除权除息处理后的价格数据。

采用了一种**请求-暂存-回调**的异步/同步协作模式：

1. **C++ 请求**：引擎需要某合约的 K 线，调用 `loadFinalHisBars`。
2. **保存上下文**：`WtRtRunner` 暂时保存 *是谁请求的*（`obj`）和 *数据回来给谁*（`cb`）。
3. **转发请求**：`WtRtRunner` 调用外部语言注册的函数指针 `_ext_fnl_bar_loader`。
4. **外部加载**：Python/C# 端执行数据查询逻辑。
5. **数据回送**：外部语言加载完成后，调用 `feedRawBars`，`WtRtRunner` 再利用第 2 步保存的上下文，通过 `_feeder_bars` 回调将数据塞回 C++ 引擎。

具体执行步骤：
* **线程安全锁**
  * 获取互斥锁 `StdUniqueLock lock(_feed_mtx);`。
  * **目的**：保护共享成员变量 `_feed_obj` 和 `_feeder_bars`。因为可能有多个线程同时触发数据加载请求，或者在数据回送（feed）尚未完成时又有新的请求进来，加锁防止上下文数据被覆盖或竞争。
* **检查外部加载器**
  * 检查 `_ext_fnl_bar_loader` 是否为 `NULL`。
  * **逻辑**：如果上层（Python端）没有注册过复权数据加载函数，说明不支持外部加载，直接返回 `false`。
* **保存回调上下文**
  * `_feed_obj = obj;`：保存数据请求发起者的句柄（通常是 `WtDataMgr` 内部的对象指针）。
  * `_feeder_bars = cb;`：保存接收数据的回调函数指针。
  * **作用**：当外部数据加载完毕调用 `feedRawBars` 时，需要知道这些数据该通过哪个函数（`cb`）传给哪个对象（`obj`）。
* **周期适配与转发**
  * 根据 C++ 内部的枚举类型 `period`，转换为外部接口约定的字符串格式：
    * `KP_DAY`  传入 `"d1"`
    * `KP_Minute1`  传入 `"m1"`
    * `KP_Minute5`  传入 `"m5"`
  * **异常处理**：如果传入了不支持的周期（如周线、年线），记录错误日志 `Unsupported period...` 并返回 `false`。
* **调用外部接口**
  * 执行 `return _ext_fnl_bar_loader(stdCode, "周期字符串");`。
  * **逻辑**：这行代码真正跨越了语言边界，控制权转交给 Python/C# 端的加载逻辑。
  * **返回值**：返回外部函数的执行结果（通常 `true` 表示找到了数据并开始推送，`false` 表示未找到或出错）。

```cpp
/**
 * @brief 加载最终历史K线数据
 * 
 * 通过外部最终K线数据加载器加载指定合约的最终历史K线数据（已复权）
 * 
 * @param obj 数据加载上下文对象指针
 * @param stdCode 标准合约代码
 * @param period K线周期类型（日线、1分钟、5分钟等）
 * @param cb 数据读取回调函数，用于接收加载的K线数据
 * @return 如果加载成功返回true，否则返回false
 */
bool WtBtRunner::loadFinalHisBars(void* obj, const char* stdCode, WTSKlinePeriod period, FuncReadBars cb)
{
	StdUniqueLock lock(_feed_mtx);  // 加锁保护数据加载过程
	if (_ext_fnl_bar_loader == NULL)  // 如果外部最终K线数据加载器未注册
		return false;  // 返回失败

	_feed_obj = obj;  // 保存数据加载上下文对象
	_feeder_bars = cb;  // 保存数据读取回调函数

	switch (period)  // 根据K线周期类型选择对应的周期字符串
	{
	case KP_DAY:  // 日线
		return _ext_fnl_bar_loader(stdCode, "d1");  // 调用外部加载器加载日线数据
	case KP_Minute1:  // 1分钟线
		return _ext_fnl_bar_loader(stdCode, "m1");  // 调用外部加载器加载1分钟线数据
	case KP_Minute5:  // 5分钟线
		return _ext_fnl_bar_loader(stdCode, "m5");  // 调用外部加载器加载5分钟线数据
	default:  // 不支持的周期类型
		{
			WTSLogger::error("Unsupported period of extended data loader");  // 记录错误日志
			return false;  // 返回失败
		}
	}
}
```

### 加载原始K线数据 loadRawHisBars
```cpp
/**
 * @brief 加载原始历史K线数据
 * 
 * 通过外部原始K线数据加载器加载指定合约的原始历史K线数据（未复权）
 * 
 * @param obj 数据加载上下文对象指针
 * @param stdCode 标准合约代码
 * @param period K线周期类型（日线、1分钟、5分钟等）
 * @param cb 数据读取回调函数，用于接收加载的K线数据
 * @return 如果加载成功返回true，否则返回false
 */
bool WtBtRunner::loadRawHisBars(void* obj, const char* stdCode, WTSKlinePeriod period, FuncReadBars cb)
{
	StdUniqueLock lock(_feed_mtx);  // 加锁保护数据加载过程
	if (_ext_raw_bar_loader == NULL)  // 如果外部原始K线数据加载器未注册
		return false;  // 返回失败

	_feed_obj = obj;  // 保存数据加载上下文对象
	_feeder_bars = cb;  // 保存数据读取回调函数

	switch (period)  // 根据K线周期类型选择对应的周期字符串
	{
	case KP_DAY:  // 日线
        return _ext_raw_bar_loader(stdCode, "d1");  // 调用外部加载器加载日线数据
	case KP_Minute1:  // 1分钟线
        return _ext_raw_bar_loader(stdCode, "m1");  // 调用外部加载器加载1分钟线数据
	case KP_Minute5:  // 5分钟线
        return _ext_raw_bar_loader(stdCode, "m5");  // 调用外部加载器加载5分钟线数据
	default:  // 不支持的周期类型
		{
			WTSLogger::error("Unsupported period of extended data loader");  // 记录错误日志
			return false;  // 返回失败
		}
	}
}
```

### 加载所有复权因子 loadAllAdjFactors
```cpp
/**
 * @brief 加载所有合约的复权因子
 * 
 * 通过外部复权因子加载器加载所有合约的复权因子数据
 * 
 * @param obj 数据加载上下文对象指针
 * @param cb 数据读取回调函数，用于接收加载的复权因子数据
 * @return 如果加载成功返回true，否则返回false
 */
bool WtBtRunner::loadAllAdjFactors(void* obj, FuncReadFactors cb)
{
	StdUniqueLock lock(_feed_mtx);  // 加锁保护数据加载过程
	if (_ext_adj_fct_loader == NULL)  // 如果外部复权因子加载器未注册
		return false;  // 返回失败

	_feed_obj = obj;  // 保存数据加载上下文对象
	_feeder_fcts = cb;  // 保存数据读取回调函数

	return _ext_adj_fct_loader("");  // 调用外部加载器加载所有合约的复权因子（空字符串表示所有合约）
}
```

### 加载复权因子 loadAdjFactors
```cpp
/**
 * @brief 加载指定合约的复权因子
 * 
 * 通过外部复权因子加载器加载指定合约的复权因子数据
 * 
 * @param obj 数据加载上下文对象指针
 * @param stdCode 标准合约代码
 * @param cb 数据读取回调函数，用于接收加载的复权因子数据
 * @return 如果加载成功返回true，否则返回false
 */
bool WtBtRunner::loadAdjFactors(void* obj, const char* stdCode, FuncReadFactors cb)
{
	StdUniqueLock lock(_feed_mtx);  // 加锁保护数据加载过程
	if (_ext_adj_fct_loader == NULL)  // 如果外部复权因子加载器未注册
		return false;  // 返回失败

	_feed_obj = obj;  // 保存数据加载上下文对象
	_feeder_fcts = cb;  // 保存数据读取回调函数

	return _ext_adj_fct_loader(stdCode);  // 调用外部加载器加载指定合约的复权因子
}
```

### 加载原始Tick数据 loadRawHisTicks
```cpp
/**
 * @brief 加载原始历史Tick数据
 * 
 * 通过外部Tick数据加载器加载指定合约在指定日期的原始历史Tick数据
 * 
 * @param obj 数据加载上下文对象指针
 * @param stdCode 标准合约代码
 * @param uDate 交易日（格式：YYYYMMDD）
 * @param cb 数据读取回调函数，用于接收加载的Tick数据
 * @return 如果加载成功返回true，否则返回false
 */
bool WtBtRunner::loadRawHisTicks(void* obj, const char* stdCode, uint32_t uDate, FuncReadTicks cb)
{
	StdUniqueLock lock(_feed_mtx);  // 加锁保护数据加载过程
	if (_ext_tick_loader == NULL)  // 如果外部Tick数据加载器未注册
		return false;  // 返回失败

	_feed_obj = obj;  // 保存数据加载上下文对象
	_feeder_ticks = cb;  // 保存数据读取回调函数

	return _ext_tick_loader(stdCode, uDate);  // 调用外部加载器加载指定合约在指定日期的Tick数据
}
```

### 是否自动转储数据 isAutoTrans
```cpp
/**
 * @brief 是否自动转储数据
 * 
 * 实现IBtDataLoader接口，返回是否自动转储数据到本地缓存
 * 
 * @return true表示自动转储，false表示不转储
 */
virtual bool isAutoTrans() override
{
    return _loader_auto_trans;  // 返回自动转储标志
}
```

## 数据推送接口

### 推送原始K线数据 feedRawBars
```cpp
/**
 * @brief 推送原始K线数据
 * 
 * 将外部加载的原始K线数据推送给回测引擎的数据加载器
 * 
 * @param bars K线数据数组指针
 * @param count K线数据数量
 */
void WtBtRunner::feedRawBars(WTSBarStruct* bars, uint32_t count)
{
	if(_ext_fnl_bar_loader == NULL && _ext_raw_bar_loader == NULL)  // 如果外部K线数据加载器未注册
	{
		WTSLogger::error("Cannot feed bars because of no extented bar loader registered.");  // 记录错误日志
		return;  // 直接返回
	}

	_feeder_bars(_feed_obj, bars, count);  // 调用数据读取回调函数，推送K线数据
}
```

### 推送原始Tick数据 feedRawTicks
```cpp
/**
 * @brief 推送原始Tick数据
 * 
 * 将外部加载的原始Tick数据推送给回测引擎的数据加载器
 * 
 * @param ticks Tick数据数组指针
 * @param count Tick数据数量
 */
void WtBtRunner::feedRawTicks(WTSTickStruct* ticks, uint32_t count)
{
	if (_ext_tick_loader == NULL)  // 如果外部Tick数据加载器未注册
	{
		WTSLogger::error("Cannot feed ticks because of no extented tick loader registered.");  // 记录错误日志
		return;  // 直接返回
	}

	_feeder_ticks(_feed_obj, ticks, count);  // 调用数据读取回调函数，推送Tick数据
}
```

### 推送复权因子数据 feedAdjFactors
```cpp
/**
 * @brief 推送复权因子数据
 * 
 * 将外部加载的复权因子数据推送给回测引擎的数据加载器
 * 
 * @param stdCode 标准合约代码
 * @param dates 复权日期数组指针
 * @param factors 复权因子数组指针
 * @param count 复权因子数量
 */
void WtBtRunner::feedAdjFactors(const char* stdCode, uint32_t* dates, double* factors, uint32_t count)
{
	if(_ext_adj_fct_loader == NULL)  // 如果外部复权因子加载器未注册
	{
		WTSLogger::error("Cannot feed adjusting factors because of no extented adjusting factor loader registered.");  // 记录错误日志
		return;  // 直接返回
	}

	_feeder_fcts(_feed_obj, stdCode, dates, factors, count);  // 调用数据读取回调函数，推送复权因子数据
}
```

## 回调函数注册接口

### 注册CTA策略回调函数 registerCtaCallbacks
```cpp
/**
 * @brief 注册CTA策略回调函数
 * 
 * 注册CTA策略相关的所有回调函数，用于接收CTA策略的各种事件通知
 * 
 * @param cbInit CTA策略初始化回调函数
 * @param cbTick CTA策略Tick更新回调函数
 * @param cbCalc CTA策略计算回调函数
 * @param cbBar CTA策略K线闭合回调函数
 * @param cbSessEvt CTA策略交易日事件回调函数
 * @param cbCalcDone CTA策略计算完成回调函数（可选）
 * @param cbCondTrigger CTA策略条件触发回调函数（可选）
 */
void WtBtRunner::registerCtaCallbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraCalcCallback cbCalc, FuncStraBarCallback cbBar, 
	FuncSessionEvtCallback cbSessEvt, FuncStraCalcCallback cbCalcDone /* = NULL */, FuncStraCondTriggerCallback cbCondTrigger /* = NULL */)
{
	_cb_cta_init = cbInit;  // 保存CTA策略初始化回调函数
	_cb_cta_tick = cbTick;  // 保存CTA策略Tick更新回调函数
	_cb_cta_calc = cbCalc;  // 保存CTA策略计算回调函数
	_cb_cta_bar = cbBar;  // 保存CTA策略K线闭合回调函数
	_cb_cta_sessevt = cbSessEvt;  // 保存CTA策略交易日事件回调函数

	_cb_cta_calc_done = cbCalcDone;  // 保存CTA策略计算完成回调函数
	_cb_cta_cond_trigger = cbCondTrigger;  // 保存CTA策略条件触发回调函数

	WTSLogger::info("Callbacks of CTA engine registration done");  // 记录注册完成日志
}
```

### 注册SEL策略回调函数 registerSelCallbacks
```cpp
/**
 * @brief 注册SEL策略回调函数
 * 
 * 注册SEL策略相关的所有回调函数，用于接收SEL策略的各种事件通知
 * 
 * @param cbInit SEL策略初始化回调函数
 * @param cbTick SEL策略Tick更新回调函数
 * @param cbCalc SEL策略计算回调函数
 * @param cbBar SEL策略K线闭合回调函数
 * @param cbSessEvt SEL策略交易日事件回调函数
 * @param cbCalcDone SEL策略计算完成回调函数（可选）
 */
void WtBtRunner::registerSelCallbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraCalcCallback cbCalc, 
	FuncStraBarCallback cbBar, FuncSessionEvtCallback cbSessEvt, FuncStraCalcCallback cbCalcDone/* = NULL*/)
{
	_cb_sel_init = cbInit;  // 保存SEL策略初始化回调函数
	_cb_sel_tick = cbTick;  // 保存SEL策略Tick更新回调函数
	_cb_sel_calc = cbCalc;  // 保存SEL策略计算回调函数
	_cb_sel_bar = cbBar;  // 保存SEL策略K线闭合回调函数
	_cb_sel_sessevt = cbSessEvt;  // 保存SEL策略交易日事件回调函数

	_cb_sel_calc_done = cbCalcDone;  // 保存SEL策略计算完成回调函数

	WTSLogger::info("Callbacks of SEL engine registration done");  // 记录注册完成日志
}
```

### 注册HFT策略回调函数 registerHftCallbacks
```cpp
/**
 * @brief 注册HFT策略回调函数
 * 
 * 注册HFT策略相关的所有回调函数，用于接收HFT策略的各种事件通知
 * 
 * @param cbInit HFT策略初始化回调函数
 * @param cbTick HFT策略Tick更新回调函数
 * @param cbBar HFT策略K线闭合回调函数
 * @param cbChnl HFT策略通道事件回调函数
 * @param cbOrd HFT策略订单回调函数
 * @param cbTrd HFT策略成交回调函数
 * @param cbEntrust HFT策略委托回调函数
 * @param cbOrdDtl HFT策略订单明细回调函数
 * @param cbOrdQue HFT策略订单队列回调函数
 * @param cbTrans HFT策略逐笔成交回调函数
 * @param cbSessEvt HFT策略交易日事件回调函数
 */
void WtBtRunner::registerHftCallbacks(FuncStraInitCallback cbInit, FuncStraTickCallback cbTick, FuncStraBarCallback cbBar,
	FuncHftChannelCallback cbChnl, FuncHftOrdCallback cbOrd, FuncHftTrdCallback cbTrd, FuncHftEntrustCallback cbEntrust, 
	FuncStraOrdDtlCallback cbOrdDtl, FuncStraOrdQueCallback cbOrdQue, FuncStraTransCallback cbTrans, FuncSessionEvtCallback cbSessEvt)
{
	_cb_hft_init = cbInit;  // 保存HFT策略初始化回调函数
	_cb_hft_tick = cbTick;  // 保存HFT策略Tick更新回调函数
	_cb_hft_bar = cbBar;  // 保存HFT策略K线闭合回调函数

	_cb_hft_chnl = cbChnl;  // 保存HFT策略通道事件回调函数
	_cb_hft_ord = cbOrd;  // 保存HFT策略订单回调函数
	_cb_hft_trd = cbTrd;  // 保存HFT策略成交回调函数
	_cb_hft_entrust = cbEntrust;  // 保存HFT策略委托回调函数

	_cb_hft_orddtl = cbOrdDtl;  // 保存HFT策略订单明细回调函数
	_cb_hft_ordque = cbOrdQue;  // 保存HFT策略订单队列回调函数
	_cb_hft_trans = cbTrans;  // 保存HFT策略逐笔成交回调函数

	_cb_hft_sessevt = cbSessEvt;  // 保存HFT策略交易日事件回调函数

	WTSLogger::info("Callbacks of HFT engine registration done");  // 记录注册完成日志
}
```

### 注册引擎事件回调函数 registerEvtCallback
```cpp
/**
 * @brief 注册引擎事件回调函数
 * 
 * 注册回测引擎的事件回调函数
 * 
 * @param cbEvt 事件回调函数指针
 */
void registerEvtCallback(FuncEventCallback cbEvt)
{
    _cb_evt = cbEvt;  // 保存事件回调函数指针
}
```

### 注册外部数据加载器 registerExtDataLoader
```cpp
/**
 * @brief 注册外部数据加载器
 * 
 * 注册外部数据加载器的回调函数
 * 
 * @param fnlBarLoader 加载最终K线数据的回调函数（已复权）
 * @param rawBarLoader 加载原始K线数据的回调函数（未复权）
 * @param fctLoader 加载复权因子的回调函数
 * @param tickLoader 加载原始Tick数据的回调函数
 * @param bAutoTrans 是否自动转储数据到本地缓存（默认true）
 */
void		registerExtDataLoader(FuncLoadFnlBars fnlBarLoader, FuncLoadRawBars rawBarLoader, FuncLoadAdjFactors fctLoader, FuncLoadRawTicks tickLoader, bool bAutoTrans = true)
{
    _ext_fnl_bar_loader = fnlBarLoader;  // 保存最终K线加载器回调函数
    _ext_raw_bar_loader = rawBarLoader;  // 保存原始K线加载器回调函数
    _ext_adj_fct_loader = fctLoader;  // 保存复权因子加载器回调函数
    _ext_tick_loader = tickLoader;  // 保存Tick加载器回调函数
    _loader_auto_trans = bAutoTrans;  // 保存自动转储标志
}
```

## 策略模拟器初始化接口

### 初始化CTA策略模拟器 initCtaMocker
```cpp
/**
 * @brief 初始化CTA策略模拟器
 * 
 * 创建并初始化CTA策略模拟器，用于回测CTA策略
 * 
 * @param name 策略名称
 * @param slippage 滑点设置（单位：最小变动价位）
 * @param hook 是否安装钩子（用于异步回测）
 * @param persistData 是否持久化数据
 * @param bIncremental 是否加载增量数据
 * @param isRatioSlp 是否使用比例滑点
 * @return 返回策略上下文ID
 */
uint32_t WtBtRunner::initCtaMocker(const char* name, int32_t slippage /* = 0 */, bool hook /* = false */, 
	bool persistData /* = true */, bool bIncremental /* = false */, bool isRatioSlp /* = false */)
{
	if(_cta_mocker)  // 如果已存在CTA模拟器
	{
		delete _cta_mocker;  // 删除旧的模拟器
		_cta_mocker = NULL;  // 重置指针
	}

	_cta_mocker = new ExpCtaMocker(&_replayer, name, slippage, persistData, &_notifier, isRatioSlp);  // 创建新的CTA模拟器
	if (bIncremental)  // 如果需要加载增量数据
	{
		_cta_mocker->load_incremental_data(name);  // 加载增量数据
	}
	if(hook) _cta_mocker->install_hook();  // 如果需要，安装钩子
	_replayer.register_sink(_cta_mocker, name);  // 将模拟器注册为数据回放器的接收器
	return _cta_mocker->id();  // 返回策略上下文ID
}
```

### 初始化HFT策略模拟器 initHftMocker
```cpp
/**
 * @brief 初始化HFT策略模拟器
 * 
 * 创建并初始化HFT策略模拟器，用于回测HFT策略
 * 
 * @param name 策略名称
 * @param hook 是否安装钩子（用于异步回测）
 * @return 返回策略上下文ID
 */
uint32_t WtBtRunner::initHftMocker(const char* name, bool hook/* = false*/)
{
	if (_hft_mocker)  // 如果已存在HFT模拟器
	{
		delete _hft_mocker;  // 删除旧的模拟器
		_hft_mocker = NULL;  // 重置指针
	}

	_hft_mocker = new ExpHftMocker(&_replayer, name);  // 创建新的HFT模拟器
	if (hook) _hft_mocker->install_hook();  // 如果需要，安装钩子
	_replayer.register_sink(_hft_mocker, name);  // 将模拟器注册为数据回放器的接收器
	return _hft_mocker->id();  // 返回策略上下文ID
}
```

### 初始化SEL策略模拟器 initSelMocker
```cpp
/**
 * @brief 初始化SEL策略模拟器
 * 
 * 创建并初始化SEL策略模拟器，用于回测SEL策略
 * 
 * @param name 策略名称
 * @param date 策略调度日期（格式：YYYYMMDD）
 * @param time 策略调度时间（格式：HHMMSS）
 * @param period 策略调度周期（如"m1"、"d1"等）
 * @param trdtpl 交易日历模板（默认为"CHINA"）
 * @param session 交易时段（默认为"TRADING"）
 * @param slippage 滑点设置（单位：最小变动价位）
 * @param isRatioSlp 是否使用比例滑点
 * @return 返回策略上下文ID
 */
uint32_t WtBtRunner::initSelMocker(const char* name, uint32_t date, uint32_t time, const char* period, 
	const char* trdtpl /* = "CHINA" */, const char* session /* = "TRADING" */, int32_t slippage /* = 0 */, bool isRatioSlp /* = false */)
{
	if (_sel_mocker)  // 如果已存在SEL模拟器
	{
		delete _sel_mocker;  // 删除旧的模拟器
		_sel_mocker = NULL;  // 重置指针
	}

	_sel_mocker = new ExpSelMocker(&_replayer, name, slippage, isRatioSlp);  // 创建新的SEL模拟器
	_replayer.register_sink(_sel_mocker, name);  // 将模拟器注册为数据回放器的接收器

	_replayer.register_task(_sel_mocker->id(), date, time, period, trdtpl, session);  // 注册策略调度任务
	return _sel_mocker->id();  // 返回策略上下文ID
}
```

### 初始化事件通知器 initEvtNotifier
```cpp
/**
 * @brief 初始化事件推送器
 * 
 * 根据配置信息初始化事件推送器
 * 
 * @param cfg 事件推送器配置（WTSVariant对象）
 * @return 如果初始化成功返回true，否则返回false
 */
bool WtBtRunner::initEvtNotifier(WTSVariant* cfg)
{
	if (cfg == NULL || cfg->type() != WTSVariant::VT_Object)  // 如果配置无效或类型不正确
		return false;  // 返回失败

	_notifier.init(cfg);  // 初始化事件推送器

	return true;  // 返回成功
}
```

## 策略事件通知接口

### 策略初始化事件通知 ctx_on_init
```cpp
/**
 * @brief 通知策略初始化事件
 * 
 * 根据引擎类型，调用对应的策略初始化回调函数，通知外部语言策略已初始化
 * 
 * @param id 策略上下文ID
 * @param eType 引擎类型（CTA、HFT或SEL）
 */
void WtBtRunner::ctx_on_init(uint32_t id, EngineType eType/*= ET_CTA*/)
{
	switch (eType)  // 根据引擎类型选择对应的回调函数
	{
	case ET_CTA: if (_cb_cta_init) _cb_cta_init(id); break;  // CTA引擎：调用CTA初始化回调
	case ET_HFT: if (_cb_hft_init) _cb_hft_init(id); break;  // HFT引擎：调用HFT初始化回调
	case ET_SEL: if (_cb_sel_init) _cb_sel_init(id); break;  // SEL引擎：调用SEL初始化回调
	default:
		break;  // 其他类型：不处理
	}
}
```

### 交易日事件通知 ctx_on_session_event
```cpp
/**
 * @brief 通知交易日事件
 * 
 * 根据引擎类型，调用对应的交易日事件回调函数，通知外部语言交易日开始或结束
 * 
 * @param id 策略上下文ID
 * @param curTDate 当前交易日（格式：YYYYMMDD）
 * @param isBegin 是否为交易日开始（true表示开始，false表示结束）
 * @param eType 引擎类型（CTA、HFT或SEL）
 */
void WtBtRunner::ctx_on_session_event(uint32_t id, uint32_t curTDate, bool isBegin /* = true */, EngineType eType /* = ET_CTA */)
{
	switch (eType)  // 根据引擎类型选择对应的回调函数
	{
	case ET_CTA: if (_cb_cta_sessevt) _cb_cta_sessevt(id, curTDate, isBegin); break;  // CTA引擎：调用CTA交易日事件回调
	case ET_HFT: if (_cb_hft_sessevt) _cb_hft_sessevt(id, curTDate, isBegin); break;  // HFT引擎：调用HFT交易日事件回调
	case ET_SEL: if (_cb_sel_sessevt) _cb_sel_sessevt(id, curTDate, isBegin); break;  // SEL引擎：调用SEL交易日事件回调
	default:
		break;  // 其他类型：不处理
	}
}
```

### Tick更新事件通知 ctx_on_tick
```cpp
/**
 * @brief 通知Tick更新事件
 * 
 * 根据引擎类型，调用对应的Tick更新回调函数，通知外部语言Tick数据已更新
 * 
 * @param id 策略上下文ID
 * @param stdCode 标准合约代码
 * @param newTick 新的Tick数据指针
 * @param eType 引擎类型（CTA、HFT或SEL）
 */
void WtBtRunner::ctx_on_tick(uint32_t id, const char* stdCode, WTSTickData* newTick, EngineType eType/*= ET_CTA*/)
{
	switch (eType)  // 根据引擎类型选择对应的回调函数
	{
	case ET_CTA: if (_cb_cta_tick) _cb_cta_tick(id, stdCode, &newTick->getTickStruct()); break;  // CTA引擎：调用CTA Tick更新回调
	case ET_HFT: if (_cb_hft_tick) _cb_hft_tick(id, stdCode, &newTick->getTickStruct()); break;  // HFT引擎：调用HFT Tick更新回调
	case ET_SEL: if (_cb_sel_tick) _cb_sel_tick(id, stdCode, &newTick->getTickStruct()); break;  // SEL引擎：调用SEL Tick更新回调
	default:
		break;  // 其他类型：不处理
	}
}
```

### 策略计算事件通知 ctx_on_calc
```cpp
/**
 * @brief 通知策略计算事件
 * 
 * 根据引擎类型，调用对应的策略计算回调函数，通知外部语言执行策略计算
 * 
 * @param id 策略上下文ID
 * @param curDate 当前日期（格式：YYYYMMDD）
 * @param curTime 当前时间（格式：HHMMSS）
 * @param eType 引擎类型（CTA或SEL）
 */
void WtBtRunner::ctx_on_calc(uint32_t id, uint32_t curDate, uint32_t curTime, EngineType eType /* = ET_CTA */)
{
	switch (eType)  // 根据引擎类型选择对应的回调函数
	{
	case ET_CTA: if (_cb_cta_calc) _cb_cta_calc(id, curDate, curTime); break;  // CTA引擎：调用CTA计算回调
	case ET_SEL: if (_cb_sel_calc) _cb_sel_calc(id, curDate, curTime); break;  // SEL引擎：调用SEL计算回调
	default:
		break;  // 其他类型：不处理
	}
}
```

### 策略计算完成事件通知 ctx_on_calc_done
```cpp
/**
 * @brief 通知策略计算完成事件
 * 
 * 根据引擎类型，调用对应的策略计算完成回调函数，通知外部语言策略计算已完成
 * 
 * @param id 策略上下文ID
 * @param curDate 当前日期（格式：YYYYMMDD）
 * @param curTime 当前时间（格式：HHMMSS）
 * @param eType 引擎类型（CTA或SEL）
 */
void WtBtRunner::ctx_on_calc_done(uint32_t id, uint32_t curDate, uint32_t curTime, EngineType eType /* = ET_CTA */)
{
	switch (eType)  // 根据引擎类型选择对应的回调函数
	{
	case ET_CTA: if (_cb_cta_calc_done) _cb_cta_calc_done(id, curDate, curTime); break;  // CTA引擎：调用CTA计算完成回调
	case ET_SEL: if (_cb_sel_calc_done) _cb_sel_calc_done(id, curDate, curTime); break;  // SEL引擎：调用SEL计算完成回调
	default:
		break;  // 其他类型：不处理
	}
}
```

### K线闭合事件通知 ctx_on_bar
```cpp
/**
 * @brief 通知K线闭合事件
 * 
 * 根据引擎类型，调用对应的K线闭合回调函数，通知外部语言K线闭合事件
 * 
 * @param id 策略上下文ID
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 * @param newBar 新生成的K线数据结构指针
 * @param eType 引擎类型（CTA、HFT或SEL）
 */
void WtBtRunner::ctx_on_bar(uint32_t id, const char* stdCode, const char* period, WTSBarStruct* newBar, EngineType eType/*= ET_CTA*/)
{
	switch (eType)  // 根据引擎类型选择对应的回调函数
	{
	case ET_CTA: if (_cb_cta_bar) _cb_cta_bar(id, stdCode, period, newBar); break;  // CTA引擎：调用CTA K线闭合回调
	case ET_HFT: if (_cb_hft_bar) _cb_hft_bar(id, stdCode, period, newBar); break;  // HFT引擎：调用HFT K线闭合回调
	case ET_SEL: if (_cb_sel_bar) _cb_sel_bar(id, stdCode, period, newBar); break;  // SEL引擎：调用SEL K线闭合回调
	default:
		break;  // 其他类型：不处理
	}
}
```

### 条件单触发事件通知 ctx_on_cond_triggered
```cpp
/**
 * @brief 通知条件触发事件
 * 
 * 根据引擎类型，调用对应的条件触发回调函数，通知外部语言条件已触发
 * 
 * @param id 策略上下文ID
 * @param stdCode 标准合约代码
 * @param target 目标价格
 * @param price 触发价格
 * @param usertag 用户标签
 * @param eType 引擎类型（目前仅支持CTA）
 */
void WtBtRunner::ctx_on_cond_triggered(uint32_t id, const char* stdCode, double target, double price, const char* usertag, EngineType eType /* = ET_CTA */)
{
	switch (eType)  // 根据引擎类型选择对应的回调函数
	{
	case ET_CTA: if (_cb_cta_cond_trigger) _cb_cta_cond_trigger(id, stdCode, target, price, usertag); break;  // CTA引擎：调用CTA条件触发回调
	default:
		break;  // 其他类型：不处理
	}
}
```

## HFT策略事件通知接口

### HFT订单队列更新事件通知 hft_on_order_queue
```cpp
/**
 * @brief 通知HFT订单队列更新事件
 * 
 * 调用HFT订单队列更新回调函数，通知外部语言订单队列数据已更新
 * 
 * @param id 策略上下文ID
 * @param stdCode 标准合约代码
 * @param newOrdQue 新的订单队列数据指针
 */
void WtBtRunner::hft_on_order_queue(uint32_t id, const char* stdCode, WTSOrdQueData* newOrdQue)
{
	if (_cb_hft_ordque)  // 如果回调函数已注册
		_cb_hft_ordque(id, stdCode, &newOrdQue->getOrdQueStruct());  // 调用订单队列更新回调
}
```

### HFT订单明细更新事件通知 hft_on_order_detail
```cpp
/**
 * @brief 通知HFT订单明细更新事件
 * 
 * 调用HFT订单明细更新回调函数，通知外部语言订单明细数据已更新
 * 
 * @param id 策略上下文ID
 * @param stdCode 标准合约代码
 * @param newOrdDtl 新的订单明细数据指针
 */
void WtBtRunner::hft_on_order_detail(uint32_t id, const char* stdCode, WTSOrdDtlData* newOrdDtl)
{
	if (_cb_hft_orddtl)  // 如果回调函数已注册
		_cb_hft_orddtl(id, stdCode, &newOrdDtl->getOrdDtlStruct());  // 调用订单明细更新回调
}
```

### HFT逐笔成交更新事件通知 hft_on_transaction
```cpp
/**
 * @brief 通知HFT逐笔成交更新事件
 * 
 * 调用HFT逐笔成交更新回调函数，通知外部语言逐笔成交数据已更新
 * 
 * @param id 策略上下文ID
 * @param stdCode 标准合约代码
 * @param newTrans 新的逐笔成交数据指针
 */
void WtBtRunner::hft_on_transaction(uint32_t id, const char* stdCode, WTSTransData* newTrans)
{
	if (_cb_hft_trans)  // 如果回调函数已注册
		_cb_hft_trans(id, stdCode, &newTrans->getTransStruct());  // 调用逐笔成交更新回调
}
```

### HFT交易通道就绪事件通知 hft_on_channel_ready
```cpp
/**
 * @brief 通知HFT通道就绪事件
 * 
 * 调用HFT通道事件回调函数，通知外部语言交易通道已就绪
 * 
 * @param cHandle 通道句柄
 * @param trader 交易通道名称
 */
void WtBtRunner::hft_on_channel_ready(uint32_t cHandle, const char* trader)
{
	if (_cb_hft_chnl)  // 如果回调函数已注册
		_cb_hft_chnl(cHandle, trader, 1000/*CHNL_EVENT_READY*/);  // 调用通道事件回调，1000表示通道就绪事件
}
```

### HFT订单状态变化事件通知 hft_on_order
```cpp
/**
 * @brief 通知HFT订单事件
 * 
 * 调用HFT订单回调函数，通知外部语言订单状态变化
 * 
 * @param cHandle 通道句柄
 * @param localid 本地订单ID
 * @param stdCode 标准合约代码
 * @param isBuy 是否为买入
 * @param totalQty 总数量
 * @param leftQty 剩余数量
 * @param price 价格
 * @param isCanceled 是否已撤销
 * @param userTag 用户标签
 */
void WtBtRunner::hft_on_order(uint32_t cHandle, WtUInt32 localid, const char* stdCode, bool isBuy, double totalQty, double leftQty, double price, bool isCanceled, const char* userTag)
{
	if (_cb_hft_ord)  // 如果回调函数已注册
		_cb_hft_ord(cHandle, localid, stdCode, isBuy, totalQty, leftQty, price, isCanceled, userTag);  // 调用订单回调
}
```

### HFT成交回报事件通知 hft_on_trade
```cpp
/**
 * @brief 通知HFT成交事件
 * 
 * 调用HFT成交回调函数，通知外部语言成交结果
 * 
 * @param cHandle 通道句柄
 * @param localid 本地订单ID
 * @param stdCode 标准合约代码
 * @param isBuy 是否为买入
 * @param vol 成交量
 * @param price 成交价格
 * @param userTag 用户标签
 */
void WtBtRunner::hft_on_trade(uint32_t cHandle, WtUInt32 localid, const char* stdCode, bool isBuy, double vol, double price, const char* userTag)
{
	if (_cb_hft_trd)  // 如果回调函数已注册
		_cb_hft_trd(cHandle, localid, stdCode, isBuy, vol, price, userTag);  // 调用成交回调
}
```

### HFT委托回报事件通知 hft_on_entrust
```cpp
/**
 * @brief 通知HFT委托事件
 * 
 * 调用HFT委托回调函数，通知外部语言委托结果
 * 
 * @param cHandle 通道句柄
 * @param localid 本地订单ID
 * @param stdCode 标准合约代码
 * @param bSuccess 是否成功
 * @param message 消息内容
 * @param userTag 用户标签
 */
void WtBtRunner::hft_on_entrust(uint32_t cHandle, WtUInt32 localid, const char* stdCode, bool bSuccess, const char* message, const char* userTag)
{
	if (_cb_hft_entrust)  // 如果回调函数已注册
		_cb_hft_entrust(cHandle, localid, stdCode, bSuccess, message, userTag);  // 调用委托回调
}
```

## 回测引擎管理接口

### 初始化回测引擎 init
```cpp
/**
 * @brief 初始化回测运行器
 * 
 * 初始化日志系统、设置工作目录和输出目录，并启用MiniDumper（Windows平台）
 * 
 * @param logProfile 日志配置文件路径或内容
 * @param isFile 是否为文件路径（true表示文件路径，false表示配置内容）
 * @param outDir 输出目录路径
 */
void WtBtRunner::init(const char* logProfile /* = "" */, bool isFile /* = true */, const char* outDir/* = "./outputs_bt"*/)
{
#ifdef _MSC_VER  // 如果是MSVC编译器
	CMiniDumper::Enable(getModuleName(), true, WtHelper::getCWD().c_str());  // 启用MiniDumper崩溃转储功能
#endif

	WTSLogger::init(logProfile, isFile);  // 初始化日志系统

	WtHelper::setInstDir(getBinDir());  // 设置安装目录
	WtHelper::setOutputDir(outDir);  // 设置输出目录
}
```

### 配置回测引擎 config

### 运行回测 run
```cpp
/**
 * @brief 运行回测
 * 
 * 启动历史数据回放，执行回测。支持同步和异步两种模式。
 * 
 * @param bNeedDump 是否需要转储数据
 * @param bAsync 是否异步运行（true表示异步，false表示同步）
 */
void WtBtRunner::run(bool bNeedDump /* = false */, bool bAsync /* = false */)
{
	if (_running)  // 如果已经在运行
		return;  // 直接返回

	_async = bAsync;  // 保存异步模式标志

	WTSLogger::info("Backtesting will run in {} mode", _async ? "async" : "sync");  // 记录运行模式日志

	if (_cta_mocker)  // 如果使用CTA模拟器
		_cta_mocker->enable_hook(_async);  // 启用或禁用钩子（用于异步回测）
	else if (_hft_mocker)  // 如果使用HFT模拟器
		_hft_mocker->enable_hook(_async);  // 启用或禁用钩子（用于异步回测）

	_replayer.prepare();  // 准备历史数据回放器
	if (!bAsync)  // 如果是同步模式
	{
		_replayer.run(bNeedDump);  // 同步运行回放器
	}
	else  // 如果是异步模式
	{
		_worker.reset(new StdThread([this, bNeedDump]() {  // 创建工作线程
			_running = true;  // 设置运行标志
			try
			{
				_replayer.run(bNeedDump);  // 在独立线程中运行回放器
			}
			catch (...)  // 捕获所有异常
			{
				WTSLogger::error("Exception raised while worker running");  // 记录错误日志
				//print_stack_trace([](const char* message) {
				//	WTSLogger::error(message);
				//});
			}
			WTSLogger::debug("Worker thread of backtest finished");  // 记录线程结束日志
			_running = false;  // 清除运行标志

		}));
	}
}
```

### 释放回测引擎 release
```cpp
/**
 * @brief 释放回测运行器
 * 
 * 停止日志系统，清理资源
 */
void WtBtRunner::release()
{
	WTSLogger::stop();  // 停止日志系统
}
```

### 停止回测 stop
```cpp
/**
 * @brief 停止回测
 * 
 * 停止正在运行的回测，等待回测完成并清理资源
 */
void WtBtRunner::stop()
{
	if (!_running)  // 如果未在运行
	{
		if (_worker)  // 如果工作线程存在
		{
			_worker->join();  // 等待工作线程结束
			_worker.reset();  // 释放工作线程
		}
		return;  // 直接返回
	}

	_replayer.stop();  // 停止历史数据回放器

	WTSLogger::debug("Notify to finish last round");  // 记录日志

	if (_cta_mocker)  // 如果使用CTA模拟器
		_cta_mocker->step_calc();  // 执行最后一次计算步骤

	if (_hft_mocker)  // 如果使用HFT模拟器
		_hft_mocker->step_tick();  // 执行最后一次Tick步骤

	WTSLogger::debug("Last round ended");  // 记录日志

	if (_worker)  // 如果工作线程存在
	{
		_worker->join();  // 等待工作线程结束
		_worker.reset();  // 释放工作线程
	}

	WTSLogger::freeAllDynLoggers();  // 释放所有动态日志记录器

	WTSLogger::debug("Backtest stopped");  // 记录停止日志
}
```

### 设置回测时间范围 set_time_range
```cpp
/**
 * @brief 设置回测时间范围
 * 
 * 手动设置回测的起始时间和结束时间
 * 
 * @param stime 起始时间（时间戳）
 * @param etime 结束时间（时间戳）
 */
void WtBtRunner::set_time_range(WtUInt64 stime, WtUInt64 etime)
{
	_replayer.set_time_range(stime, etime);  // 设置历史数据回放器的时间范围

	WTSLogger::info("Backtest time range is set to be [{},{}] mannually", stime, etime);  // 记录日志
}
```

### 启用/禁用Tick回放 enable_tick
```cpp
/**
 * @brief 启用或禁用Tick数据回放
 * 
 * 控制是否回放Tick级别的数据
 * 
 * @param bEnabled 是否启用（true表示启用，false表示禁用）
 */
void WtBtRunner::enable_tick(bool bEnabled /* = true */)
{
	_replayer.enable_tick(bEnabled);  // 设置历史数据回放器的Tick回放标志

	WTSLogger::info("Tick data replaying is {}", bEnabled ? "enabled" : "disabled");  // 记录日志
}
```

### 清空缓存 clear_cache
```cpp
/**
 * @brief 清除缓存
 * 
 * 清除历史数据回放器的缓存数据
 */
void WtBtRunner::clear_cache()
{
	_replayer.clear_cache();  // 清除历史数据回放器的缓存
}
```

### 获取原始标准代码 get_raw_stdcode
```cpp
/**
 * @brief 获取原始合约代码
 * 
 * 将标准合约代码转换为原始合约代码
 * 
 * @param stdCode 标准合约代码
 * @return 返回原始合约代码字符串（线程局部存储）
 */
const char* WtBtRunner::get_raw_stdcode(const char* stdCode)
{
	static thread_local std::string s;  // 线程局部存储的字符串
	s = _replayer.get_rawcode(stdCode);  // 获取原始合约代码
	return s.c_str();  // 返回C字符串
}
```

## 访问器接口

### 获取CTA模拟器对象 cta_mocker
```cpp
/**
 * @brief 获取CTA模拟器对象
 * 
 * 返回CTA策略模拟器对象的指针
 * 
 * @return CTA模拟器对象指针（如果不存在则返回NULL）
 */
inline CtaMocker*		cta_mocker() { return _cta_mocker; }
```

### 获取SEL模拟器对象 sel_mocker
```cpp
/**
 * @brief 获取SEL模拟器对象
 * 
 * 返回SEL策略模拟器对象的指针
 * 
 * @return SEL模拟器对象指针（如果不存在则返回NULL）
 */
inline SelMocker*		sel_mocker() { return _sel_mocker; }
```

### 获取HFT模拟器对象 hft_mocker
```cpp
/**
 * @brief 获取HFT模拟器对象
 * 
 * 返回HFT策略模拟器对象的指针
 * 
 * @return HFT模拟器对象指针（如果不存在则返回NULL）
 */
inline HftMocker*		hft_mocker() { return _hft_mocker; }
```

### 获取历史数据回放器对象 replayer
```cpp
/**
 * @brief 获取历史数据回放器对象
 * 
 * 返回历史数据回放器对象的引用
 * 
 * @return 历史数据回放器对象的引用
 */
inline HisDataReplayer&	replayer() { return _replayer; }
```

### 是否异步模式 isAsync
```cpp
/**
 * @brief 是否异步模式
 * 
 * 返回当前是否处于异步回测模式
 * 
 * @return true表示异步模式，false表示同步模式
 */
inline bool	isAsync() const { return _async; }
```

## 引擎事件处理接口

### 引擎初始化事件处理 on_initialize_event
```cpp
/**
 * @brief 引擎初始化事件处理
 * 
 * 当回测引擎初始化完成时调用，触发引擎初始化事件回调
 */
inline void on_initialize_event()
{
    if (_cb_evt)  // 如果事件回调函数已注册
        _cb_evt(EVENT_ENGINE_INIT, 0, 0);  // 触发引擎初始化事件回调
}
```

### 引擎调度事件处理 on_schedule_event
```cpp
/**
 * @brief 引擎调度事件处理
 * 
 * 当回测引擎按时间调度时调用，触发引擎调度事件回调
 * 
 * @param uDate 当前日期（格式：YYYYMMDD）
 * @param uTime 当前时间（格式：HHMMSS）
 */
inline void on_schedule_event(uint32_t uDate, uint32_t uTime)
{
    if (_cb_evt)  // 如果事件回调函数已注册
        _cb_evt(EVENT_ENGINE_SCHDL, uDate, uTime);  // 触发引擎调度事件回调
}
```

### 交易日事件处理 on_session_event
```cpp
/**
 * @brief 交易日事件处理
 * 
 * 当交易日开始或结束时调用，触发交易日事件回调
 * 
 * @param uDate 当前交易日（格式：YYYYMMDD）
 * @param isBegin true表示交易日开始，false表示交易日结束（默认true）
 */
inline void on_session_event(uint32_t uDate, bool isBegin = true)
{
    if (_cb_evt)  // 如果事件回调函数已注册
    {
        _cb_evt(isBegin ? EVENT_SESSION_BEGIN : EVENT_SESSION_END, uDate, 0);  // 触发交易日开始或结束事件回调
    }
}
```

### 回测结束事件处理 on_backtest_end
```cpp
/**
 * @brief 回测结束事件处理
 * 
 * 当回测完成时调用，触发回测结束事件回调
 */
inline void on_backtest_end()
{
    if (_cb_evt)  // 如果事件回调函数已注册
        _cb_evt(EVENT_BACKTEST_END, 0, 0);  // 触发回测结束事件回调
}
```

# 扩展模拟器层

## CTA策略扩展模拟器 ExpCtaMocker.h/cpp
```cpp
class ExpCtaMocker : public CtaMocker
```

### 策略初始化事件处理函数 on_init
```cpp
/**
 * @brief 策略初始化事件处理函数实现
 * 
 * 当策略初始化完成时调用此函数，执行以下操作：
 * 1. 调用基类CtaMocker::on_init()，执行基类的初始化逻辑
 * 2. 通过WtBtRunner通知外部语言策略已初始化完成（ctx_on_init）
 * 3. 触发引擎初始化事件（on_initialize_event），通知外部语言引擎已初始化
 */
void ExpCtaMocker::on_init()
{
	CtaMocker::on_init();  // 调用基类的初始化逻辑，完成策略的基础初始化工作

	// 向外部回调：通知外部语言策略已初始化完成
	// _context_id是策略上下文ID，用于标识策略实例；ET_CTA表示CTA引擎类型
	getRunner().ctx_on_init(_context_id, ET_CTA);

	// 触发引擎初始化事件，通知外部语言回测引擎已初始化完成
	getRunner().on_initialize_event();
}
```

### 交易日开始事件处理函数 on_session_begin
```cpp
/**
 * @brief 交易日开始事件处理函数实现
 * 
 * 当新的交易日开始时调用此函数，执行以下操作：
 * 1. 调用基类CtaMocker::on_session_begin()，执行基类的交易日开始逻辑
 * 2. 通过WtBtRunner通知外部语言策略的交易日已开始（ctx_on_session_event）
 * 3. 触发引擎交易日开始事件（on_session_event），通知外部语言引擎交易日已开始
 * 
 * @param uCurDate 当前交易日（格式：YYYYMMDD，如20230330表示2023年3月30日）
 */
void ExpCtaMocker::on_session_begin(uint32_t uCurDate)
{
	CtaMocker::on_session_begin(uCurDate);  // 调用基类的交易日开始逻辑，完成策略的交易日初始化工作

	// 通知外部语言策略的交易日已开始
	// 参数：策略上下文ID、当前交易日、true表示交易日开始、ET_CTA表示CTA引擎类型
	getRunner().ctx_on_session_event(_context_id, uCurDate, true, ET_CTA);
	
	// 触发引擎交易日开始事件，通知外部语言回测引擎的交易日已开始
	getRunner().on_session_event(uCurDate, true);
}
```

### 交易日结束事件处理函数 on_session_end
```cpp
/**
 * @brief 交易日结束事件处理函数实现
 * 
 * 当交易日结束时调用此函数，执行以下操作：
 * 1. 先通知外部语言策略的交易日已结束（ctx_on_session_event）
 * 2. 触发引擎交易日结束事件（on_session_event），通知外部语言引擎交易日已结束
 * 3. 最后调用基类CtaMocker::on_session_end()，执行基类的交易日结束逻辑
 * 
 * 注意：这里先通知外部语言，再调用基类，是为了确保外部语言可以在基类清理资源之前进行必要的处理。
 * 
 * @param uCurDate 当前交易日（格式：YYYYMMDD）
 */
void ExpCtaMocker::on_session_end(uint32_t uCurDate)
{
	// 先通知外部语言策略的交易日已结束
	// 参数：策略上下文ID、当前交易日、false表示交易日结束、ET_CTA表示CTA引擎类型
	getRunner().ctx_on_session_event(_context_id, uCurDate, false, ET_CTA);
	
	// 触发引擎交易日结束事件，通知外部语言回测引擎的交易日已结束
	getRunner().on_session_event(uCurDate, false);

	// 最后调用基类的交易日结束逻辑，完成策略的交易日清理工作
	CtaMocker::on_session_end(uCurDate);
}
```

### Tick数据更新事件处理函数 on_tick_updated
```cpp
/**
 * @brief Tick数据更新事件处理函数实现
 * 
 * 当订阅的合约有新的Tick数据时调用此函数，执行以下操作：
 * 1. 检查是否订阅了该合约（通过_tick_subs查找），如果未订阅则直接返回
 * 2. 调用基类CtaMocker::on_tick_updated()，执行基类的Tick更新逻辑
 * 3. 通过WtBtRunner通知外部语言有新的Tick数据（ctx_on_tick）
 * 
 * @param stdCode 标准合约代码（如"SHFE.rb2305"）
 * @param newTick 新的Tick数据指针，包含最新价格、成交量、持仓量等信息
 */
void ExpCtaMocker::on_tick_updated(const char* stdCode, WTSTickData* newTick)
{
	// 检查是否订阅了该合约的Tick数据
	// _tick_subs是基类CtaMocker的成员变量，存储已订阅的合约集合
	auto it = _tick_subs.find(stdCode);
	if (it == _tick_subs.end())  // 如果未订阅该合约，直接返回，不处理
		return;

	// 调用基类的Tick更新逻辑，更新策略内部状态
	CtaMocker::on_tick_updated(stdCode, newTick);
	
	// 通知外部语言有新的Tick数据
	// 参数：策略上下文ID、标准合约代码、新的Tick数据指针、ET_CTA表示CTA引擎类型
	getRunner().ctx_on_tick(_context_id, stdCode, newTick, ET_CTA);
}
```

### K线闭合事件处理函数 on_bar_close
```cpp
/**
 * @brief K线闭合事件处理函数实现
 * 
 * 当订阅的K线周期完成并生成新的K线时调用此函数，执行以下操作：
 * 1. 调用基类CtaMocker::on_bar_close()，执行基类的K线闭合逻辑
 * 2. 通过WtBtRunner通知外部语言有新的K线闭合事件（ctx_on_bar）
 * 
 * @param code 标准合约代码（如"SHFE.rb2305"）
 * @param period K线周期（如"m1"表示1分钟K线，"d1"表示日线）
 * @param newBar 新生成的K线数据指针，包含开盘价、最高价、最低价、收盘价、成交量等信息
 */
void ExpCtaMocker::on_bar_close(const char* code, const char* period, WTSBarStruct* newBar)
{
	// 调用基类的K线闭合逻辑，更新策略内部状态，触发策略计算
	CtaMocker::on_bar_close(code, period, newBar);

	// 要向外部回调：通知外部语言有新的K线闭合事件
	// 参数：策略上下文ID、标准合约代码、K线周期、新生成的K线数据指针、ET_CTA表示CTA引擎类型
	getRunner().ctx_on_bar(_context_id, code, period, newBar, ET_CTA);
}
```

### 策略计算事件处理函数 on_calculate
```cpp
/**
 * @brief 策略计算事件处理函数实现
 * 
 * 当策略需要执行计算逻辑时调用此函数（通常在K线闭合或定时触发时），执行以下操作：
 * 1. 调用基类CtaMocker::on_calculate()，执行基类的计算逻辑
 * 2. 通过WtBtRunner通知外部语言策略需要计算（ctx_on_calc）
 * 
 * @param curDate 当前日期（格式：YYYYMMDD）
 * @param curTime 当前时间（格式：HHMMSS，如143000表示14:30:00）
 */
void ExpCtaMocker::on_calculate(uint32_t curDate, uint32_t curTime)
{
	// 调用基类的计算逻辑，执行策略的计算函数
	CtaMocker::on_calculate(curDate, curTime);
	
	// 通知外部语言策略需要计算
	// 参数：策略上下文ID、当前日期、当前时间、ET_CTA表示CTA引擎类型
	getRunner().ctx_on_calc(_context_id, curDate, curTime, ET_CTA);
}
```

### 策略计算完成事件处理函数 on_calculate_done
```cpp
/**
 * @brief 策略计算完成事件处理函数实现
 * 
 * 当策略计算逻辑执行完毕后调用此函数，执行以下操作：
 * 1. 调用基类CtaMocker::on_calculate_done()，执行基类的计算完成逻辑
 * 2. 通过WtBtRunner通知外部语言策略计算已完成（ctx_on_calc_done）
 * 3. 触发引擎调度事件（on_schedule_event），通知外部语言引擎已完成一次调度
 * 
 * @param curDate 当前日期（格式：YYYYMMDD）
 * @param curTime 当前时间（格式：HHMMSS）
 */
void ExpCtaMocker::on_calculate_done(uint32_t curDate, uint32_t curTime)
{
	// 调用基类的计算完成逻辑，完成策略计算后的清理工作
	CtaMocker::on_calculate_done(curDate, curTime);
	
	// 通知外部语言策略计算已完成
	// 参数：策略上下文ID、当前日期、当前时间、ET_CTA表示CTA引擎类型
	getRunner().ctx_on_calc_done(_context_id, curDate, curTime, ET_CTA);

	// 触发引擎调度事件，通知外部语言回测引擎已完成一次调度
	getRunner().on_schedule_event(curDate, curTime);
}
```

### 回测结束事件处理函数 on_bactest_end
```cpp
/**
 * @brief 回测结束事件处理函数实现
 * 
 * 当回测完成时调用此函数，通过WtBtRunner通知外部语言回测已结束。
 * 
 * 注意：此函数不调用基类实现，因为基类可能没有此函数或实现不同。
 */
void ExpCtaMocker::on_bactest_end()
{
	// 触发回测结束事件，通知外部语言回测已完成
	getRunner().on_backtest_end();
}
```

### 条件单触发事件处理函数 on_condition_triggered
```cpp
/**
 * @brief 条件单触发事件处理函数实现
 * 
 * 当策略设置的条件单被触发时调用此函数，通过WtBtRunner通知外部语言条件单已触发。
 * 
 * @param stdCode 标准合约代码
 * @param target 目标持仓数量（正数表示多头，负数表示空头）
 * @param price 触发价格（条件单触发时的价格）
 * @param usertag 用户标签（用于标识该条件单，由策略在设置条件单时指定）
 */
void ExpCtaMocker::on_condition_triggered(const char* stdCode, double target, double price, const char* usertag)
{
	// 通知外部语言条件单已触发
	// 参数：策略上下文ID、标准合约代码、目标持仓数量、触发价格、用户标签、ET_CTA表示CTA引擎类型
	getRunner().ctx_on_cond_triggered(_context_id, stdCode, target, price, usertag, ET_CTA);
}
```

## HFT策略扩展模拟器 ExpHftMocker.h/cpp
```cpp
class ExpHftMocker : public HftMocker
```

### K线闭合事件处理函数 on_bar
```cpp
/**
 * @brief K线闭合事件处理函数实现
 * 
 * 当订阅的K线周期完成并生成新的K线时调用此函数，执行以下操作：
 * 1. 检查K线数据是否有效，如果为NULL则直接返回
 * 2. 将K线周期转换为标准格式（如"m5"表示5分钟K线，"d1"表示日线）
 * 3. 调用基类HftMocker::on_bar()，执行基类的K线闭合逻辑
 * 4. 通过WtBtRunner通知外部语言有新的K线闭合事件（ctx_on_bar）
 * 
 * @param stdCode 标准合约代码（如"SHFE.rb2305"）
 * @param period K线周期（如"m"表示分钟，"d"表示日）
 * @param times K线倍数（如period为"m"时，times为5表示5分钟K线）
 * @param newBar 新生成的K线数据指针，包含开盘价、最高价、最低价、收盘价、成交量等信息
 */
void ExpHftMocker::on_bar(const char* stdCode, const char* period, uint32_t times, WTSBarStruct* newBar)
{
	// 检查K线数据是否有效
	if (newBar == NULL)  // 如果K线数据为空，直接返回，不处理
		return;

	// 将K线周期转换为标准格式
	// 如果周期以'd'开头（如"d"），则格式化为"d{times}"（如"d1"表示日线）
	// 否则格式化为"m{times}"（如"m5"表示5分钟K线）
	std::string realPeriod;
	if (period[0] == 'd')  // 日线周期
		realPeriod = StrUtil::printf("%s%u", period, times);  // 格式化为"d{times}"
	else  // 分钟周期
		realPeriod = StrUtil::printf("m%u", times);  // 格式化为"m{times}"

	// 调用基类的K线闭合逻辑，更新策略内部状态，触发策略计算
	HftMocker::on_bar(stdCode, period, times, newBar);

	// 通知外部语言有新的K线闭合事件
	// 参数：策略上下文ID、标准合约代码、标准格式的K线周期、新生成的K线数据指针、ET_HFT表示HFT引擎类型
	getRunner().ctx_on_bar(_context_id, stdCode, realPeriod.c_str(), newBar, ET_HFT);
}
```

### 交易通道就绪事件处理函数 on_channel_ready
```cpp
/**
 * @brief 交易通道就绪事件处理函数实现
 * 
 * 当HFT策略的交易通道连接成功并准备就绪时调用此函数，执行以下操作：
 * 1. 调用基类HftMocker::on_channel_ready()，执行基类的通道就绪逻辑
 * 2. 通过WtBtRunner通知外部语言交易通道已就绪（hft_on_channel_ready）
 * 
 * 注意：交易通道ID参数为空字符串，因为回测模式下没有实际的交易通道。
 */
void ExpHftMocker::on_channel_ready()
{
	// 调用基类的通道就绪逻辑，完成交易通道的初始化工作
	HftMocker::on_channel_ready();

	// 通知外部语言交易通道已就绪
	// 参数：策略上下文ID、交易通道ID（回测模式下为空字符串）
	getRunner().hft_on_channel_ready(_context_id, "");
}
```

### 委托回报事件处理函数 on_entrust
```cpp
/**
 * @brief 委托回报事件处理函数实现
 * 
 * 当HFT策略的委托单提交后收到回报时调用此函数，执行以下操作：
 * 1. 调用基类HftMocker::on_entrust()，执行基类的委托回报逻辑
 * 2. 通过WtBtRunner通知外部语言委托回报信息（hft_on_entrust）
 * 
 * @param localid 本地订单ID（下单时返回的订单ID）
 * @param stdCode 标准合约代码
 * @param bSuccess 是否成功，true表示委托成功，false表示委托失败
 * @param message 返回消息（如果失败，包含失败原因）
 * @param userTag 用户标签（用于标识该笔交易）
 */
void ExpHftMocker::on_entrust(uint32_t localid, const char* stdCode, bool bSuccess, const char* message, const char* userTag)
{
	// 调用基类的委托回报逻辑，更新订单状态
	HftMocker::on_entrust(localid, stdCode, bSuccess, message, userTag);

	// 通知外部语言委托回报信息
	// 参数：策略上下文ID、本地订单ID、标准合约代码、是否成功、返回消息、用户标签
	getRunner().hft_on_entrust(_context_id, localid, stdCode, bSuccess, message, userTag);
}
```

### 策略初始化事件处理函数 on_init
```cpp
/**
 * @brief 策略初始化事件处理函数实现
 * 
 * 当策略初始化完成时调用此函数，执行以下操作：
 * 1. 调用基类HftMocker::on_init()，执行基类的初始化逻辑
 * 2. 通过WtBtRunner通知外部语言策略已初始化完成（ctx_on_init）
 * 3. 触发引擎初始化事件（on_initialize_event），通知外部语言引擎已初始化
 */
void ExpHftMocker::on_init()
{
	// 调用基类的初始化逻辑，完成策略的基础初始化工作
	HftMocker::on_init();

	// 通知外部语言策略已初始化完成
	// 参数：策略上下文ID、ET_HFT表示HFT引擎类型
	getRunner().ctx_on_init(_context_id, ET_HFT);

	// 触发引擎初始化事件，通知外部语言回测引擎已初始化完成
	getRunner().on_initialize_event();
}
```

### 交易日开始事件处理函数 on_session_begin
```cpp
/**
 * @brief 交易日开始事件处理函数实现
 * 
 * 当新的交易日开始时调用此函数，执行以下操作：
 * 1. 调用基类HftMocker::on_session_begin()，执行基类的交易日开始逻辑
 * 2. 通过WtBtRunner通知外部语言策略的交易日已开始（ctx_on_session_event）
 * 3. 触发引擎交易日开始事件（on_session_event），通知外部语言引擎交易日已开始
 * 
 * @param uDate 当前交易日（格式：YYYYMMDD）
 */
void ExpHftMocker::on_session_begin(uint32_t uDate)
{
	// 调用基类的交易日开始逻辑，完成策略的交易日初始化工作
	HftMocker::on_session_begin(uDate);

	// 通知外部语言策略的交易日已开始
	// 参数：策略上下文ID、当前交易日、true表示交易日开始、ET_HFT表示HFT引擎类型
	getRunner().ctx_on_session_event(_context_id, uDate, true, ET_HFT);
	
	// 触发引擎交易日开始事件，通知外部语言回测引擎的交易日已开始
	getRunner().on_session_event(uDate, true);
}
```

### 交易日结束事件处理函数 on_session_end
```cpp
/**
 * @brief 交易日结束事件处理函数实现
 * 
 * 当交易日结束时调用此函数，执行以下操作：
 * 1. 调用基类HftMocker::on_session_end()，执行基类的交易日结束逻辑
 * 2. 通过WtBtRunner通知外部语言策略的交易日已结束（ctx_on_session_event）
 * 3. 触发引擎交易日结束事件（on_session_event），通知外部语言引擎交易日已结束
 * 
 * @param uDate 当前交易日（格式：YYYYMMDD）
 */
void ExpHftMocker::on_session_end(uint32_t uDate)
{
	// 调用基类的交易日结束逻辑，完成策略的交易日清理工作
	HftMocker::on_session_end(uDate);

	// 通知外部语言策略的交易日已结束
	// 参数：策略上下文ID、当前交易日、false表示交易日结束、ET_HFT表示HFT引擎类型
	getRunner().ctx_on_session_event(_context_id, uDate, false, ET_HFT);
	
	// 触发引擎交易日结束事件，通知外部语言回测引擎的交易日已结束
	getRunner().on_session_event(uDate, false);
}
```

### 回测结束事件处理函数 on_bactest_end
```cpp
/**
 * @brief 回测结束事件处理函数实现
 * 
 * 当回测完成时调用此函数，通过WtBtRunner通知外部语言回测已结束。
 * 
 * 注意：此函数不调用基类实现，因为基类可能没有此函数或实现不同。
 */
void ExpHftMocker::on_bactest_end()
{
	// 触发回测结束事件，通知外部语言回测已完成
	getRunner().on_backtest_end();
}
```

### 订单状态变化事件处理函数 on_order
```cpp
/**
 * @brief 订单状态变化事件处理函数实现
 * 
 * 当HFT策略的订单状态发生变化时调用此函数，执行以下操作：
 * 1. 调用基类HftMocker::on_order()，执行基类的订单状态变化逻辑
 * 2. 通过WtBtRunner通知外部语言订单状态变化信息（hft_on_order）
 * 
 * @param localid 本地订单ID
 * @param stdCode 标准合约代码
 * @param isBuy true表示买入订单，false表示卖出订单
 * @param totalQty 订单总数量
 * @param leftQty 剩余未成交数量
 * @param price 订单价格
 * @param isCanceled 是否已撤单，true表示已撤单，false表示未撤单
 * @param userTag 用户标签
 */
void ExpHftMocker::on_order(uint32_t localid, const char* stdCode, bool isBuy, double totalQty, double leftQty, double price, bool isCanceled, const char* userTag)
{
	// 调用基类的订单状态变化逻辑，更新订单状态
	HftMocker::on_order(localid, stdCode, isBuy, totalQty, leftQty, price, isCanceled, userTag);

	// 通知外部语言订单状态变化信息
	// 参数：策略上下文ID、本地订单ID、标准合约代码、是否买入、订单总数量、剩余未成交数量、订单价格、是否已撤单、用户标签
	getRunner().hft_on_order(_context_id, localid, stdCode, isBuy, totalQty, leftQty, price, isCanceled, userTag);
}
```

### Tick数据更新事件处理函数 on_tick_updated
```cpp
/**
 * @brief Tick数据更新事件处理函数实现
 * 
 * 当订阅的合约有新的Tick数据时调用此函数，执行以下操作：
 * 1. 检查是否订阅了该合约（通过_tick_subs查找），如果未订阅则直接返回
 * 2. 调用基类HftMocker::on_tick_updated()，执行基类的Tick更新逻辑
 * 3. 通过WtBtRunner通知外部语言有新的Tick数据（ctx_on_tick）
 * 
 * @param stdCode 标准合约代码
 * @param newTick 新的Tick数据指针，包含最新价格、成交量、持仓量等信息
 */
void ExpHftMocker::on_tick_updated(const char* stdCode, WTSTickData* newTick)
{
	// 检查是否订阅了该合约的Tick数据
	// _tick_subs是基类HftMocker的成员变量，存储已订阅的合约集合
	auto it = _tick_subs.find(stdCode);
	if (it == _tick_subs.end())  // 如果未订阅该合约，直接返回，不处理
		return;

	// 调用基类的Tick更新逻辑，更新策略内部状态
	HftMocker::on_tick_updated(stdCode, newTick);
	
	// 通知外部语言有新的Tick数据
	// 参数：策略上下文ID、标准合约代码、新的Tick数据指针、ET_HFT表示HFT引擎类型
	getRunner().ctx_on_tick(_context_id, stdCode, newTick, ET_HFT);
}
```

### 订单队列更新事件处理函数 on_ordque_updated
```cpp
/**
 * @brief 订单队列更新事件处理函数实现
 * 
 * 当订阅的合约有新的订单队列数据（Level2行情）时调用此函数，
 * 直接通过WtBtRunner通知外部语言有新的订单队列数据。
 * 
 * 注意：此函数不调用基类实现，因为基类可能没有此函数或实现不同，直接转发给外部语言。
 * 
 * @param stdCode 标准合约代码
 * @param newOrdQue 新的订单队列数据指针，包含买卖盘口信息（买一、买二、卖一、卖二等）
 */
void ExpHftMocker::on_ordque_updated(const char* stdCode, WTSOrdQueData* newOrdQue)
{
	// 直接通知外部语言有新的订单队列数据（Level2行情）
	// 参数：策略上下文ID、标准合约代码、新的订单队列数据指针
	getRunner().hft_on_order_queue(_context_id, stdCode, newOrdQue);
}
```

### 订单明细更新事件处理函数 on_orddtl_updated
```cpp
/**
 * @brief 订单明细更新事件处理函数实现
 * 
 * 当订阅的合约有新的订单明细数据（Level2行情）时调用此函数，
 * 直接通过WtBtRunner通知外部语言有新的订单明细数据。
 * 
 * 注意：此函数不调用基类实现，因为基类可能没有此函数或实现不同，直接转发给外部语言。
 * 
 * @param stdCode 标准合约代码
 * @param newOrdDtl 新的订单明细数据指针，包含订单簿信息（每个价位的订单数量等）
 */
void ExpHftMocker::on_orddtl_updated(const char* stdCode, WTSOrdDtlData* newOrdDtl)
{
	// 直接通知外部语言有新的订单明细数据（Level2行情）
	// 参数：策略上下文ID、标准合约代码、新的订单明细数据指针
	getRunner().hft_on_order_detail(_context_id, stdCode, newOrdDtl);
}
```

### 逐笔成交更新事件处理函数 on_trans_updated
```cpp
/**
 * @brief 逐笔成交更新事件处理函数实现
 * 
 * 当订阅的合约有新的逐笔成交数据（Level2行情）时调用此函数，
 * 直接通过WtBtRunner通知外部语言有新的逐笔成交数据。
 * 
 * 注意：此函数不调用基类实现，因为基类可能没有此函数或实现不同，直接转发给外部语言。
 * 
 * @param stdCode 标准合约代码
 * @param newTrans 新的逐笔成交数据指针，包含每笔成交的详细信息（价格、数量、方向等）
 */
void ExpHftMocker::on_trans_updated(const char* stdCode, WTSTransData* newTrans)
{
	// 直接通知外部语言有新的逐笔成交数据（Level2行情）
	// 参数：策略上下文ID、标准合约代码、新的逐笔成交数据指针
	getRunner().hft_on_transaction(_context_id, stdCode, newTrans);
}
```

### 成交回报事件处理函数 on_trade
```cpp
/**
 * @brief 成交回报事件处理函数实现
 * 
 * 当HFT策略的订单有成交回报时调用此函数，执行以下操作：
 * 1. 调用基类HftMocker::on_trade()，执行基类的成交回报逻辑
 * 2. 通过WtBtRunner通知外部语言成交回报信息（hft_on_trade）
 * 
 * @param localid 本地订单ID
 * @param stdCode 标准合约代码
 * @param isBuy true表示买入成交，false表示卖出成交
 * @param vol 成交数量
 * @param price 成交价格
 * @param userTag 用户标签
 */
void ExpHftMocker::on_trade(uint32_t localid, const char* stdCode, bool isBuy, double vol, double price, const char* userTag)
{
	// 调用基类的成交回报逻辑，更新持仓和资金
	HftMocker::on_trade(localid, stdCode, isBuy, vol, price, userTag);

	// 通知外部语言成交回报信息
	// 参数：策略上下文ID、本地订单ID、标准合约代码、是否买入、成交数量、成交价格、用户标签
	getRunner().hft_on_trade(_context_id, localid, stdCode, isBuy, vol, price, userTag);
}
```

## SEL策略扩展模拟器 ExpSelMocker.h/cpp
```cpp
class ExpSelMocker : public SelMocker
```

### 策略初始化事件处理函数 on_init
```cpp
/**
 * @brief 策略初始化事件处理
 * 
 * 调用基类的初始化方法，然后通知外部语言策略已初始化
 */
void ExpSelMocker::on_init()
{
	SelMocker::on_init();  // 调用基类的初始化方法

	//向外部回调
	getRunner().ctx_on_init(_context_id, ET_SEL);  // 通知外部语言策略已初始化

	getRunner().on_initialize_event();  // 通知外部语言引擎初始化事件
}
```

### 交易日开始事件处理函数 on_session_begin
```cpp
/**
 * @brief 交易日开始事件处理
 * 
 * 调用基类的交易日开始方法，然后通知外部语言交易日开始
 * 
 * @param uDate 当前交易日（格式：YYYYMMDD）
 */
void ExpSelMocker::on_session_begin(uint32_t uDate)
{
	SelMocker::on_session_begin(uDate);  // 调用基类的交易日开始方法

	getRunner().ctx_on_session_event(_context_id, uDate, true, ET_SEL);  // 通知外部语言交易日开始
	getRunner().on_session_event(uDate, true);  // 通知外部语言引擎交易日开始事件
}
```

### 交易日结束事件处理函数 on_session_end
```cpp
/**
 * @brief 交易日结束事件处理
 * 
 * 调用基类的交易日结束方法，然后通知外部语言交易日结束
 * 
 * @param uDate 当前交易日（格式：YYYYMMDD）
 */
void ExpSelMocker::on_session_end(uint32_t uDate)
{
	SelMocker::on_session_end(uDate);  // 调用基类的交易日结束方法

	getRunner().ctx_on_session_event(_context_id, uDate, false, ET_SEL);  // 通知外部语言交易日结束
	getRunner().on_session_event(uDate, false);  // 通知外部语言引擎交易日结束事件
}
```

### 回测结束事件处理函数 on_bactest_end
```cpp
/**
 * @brief 回测结束事件处理
 * 
 * 通知外部语言回测已结束
 */
void ExpSelMocker::on_bactest_end()
{
	getRunner().on_backtest_end();  // 通知外部语言回测已结束
}
```

### Tick数据更新事件处理函数 on_tick_updated
```cpp
/**
 * @brief Tick更新事件处理
 * 
 * 检查合约是否已订阅，如果已订阅则通知外部语言Tick更新
 * 
 * @param stdCode 标准合约代码
 * @param newTick 新的Tick数据指针
 */
void ExpSelMocker::on_tick_updated(const char* stdCode, WTSTickData* newTick)
{
	auto it = _tick_subs.find(stdCode);  // 查找合约是否已订阅
	if (it == _tick_subs.end())  // 如果未订阅
		return;  // 直接返回

	//向外部回调
	getRunner().ctx_on_tick(_context_id, stdCode, newTick, ET_SEL);  // 通知外部语言Tick更新
}
```

### K线闭合事件处理函数 on_bar_close
```cpp
/**
 * @brief K线闭合事件处理
 * 
 * 调用基类的K线闭合方法，然后通知外部语言K线闭合
 * 
 * @param stdCode 标准合约代码
 * @param period K线周期（如"m1"、"d1"等）
 * @param newBar 新生成的K线数据结构指针
 */
void ExpSelMocker::on_bar_close(const char* stdCode, const char* period, WTSBarStruct* newBar)
{
	SelMocker::on_bar_close(stdCode, period, newBar);  // 调用基类的K线闭合方法
	//要向外部回调
	getRunner().ctx_on_bar(_context_id, stdCode, period, newBar, ET_SEL);  // 通知外部语言K线闭合
}
```

### 策略调度事件处理函数 on_strategy_schedule
```cpp
/**
 * @brief 策略调度事件处理
 * 
 * 调用基类的策略调度方法，然后通知外部语言执行策略计算，并触发调度事件
 * 
 * @param curDate 当前日期（格式：YYYYMMDD）
 * @param curTime 当前时间（格式：HHMMSS）
 */
void ExpSelMocker::on_strategy_schedule(uint32_t curDate, uint32_t curTime)
{
	SelMocker::on_strategy_schedule(curDate, curTime);  // 调用基类的策略调度方法

	//向外部回调
	getRunner().ctx_on_calc(_context_id, curDate, curTime, ET_SEL);  // 通知外部语言执行策略计算

	getRunner().on_schedule_event(curDate, curTime);  // 通知外部语言调度事件
}
```